# GPT gate-engine implementation ED-Pipeline

_This notebook contains the gate-first baseline (ops + 0/1h troponin), with CONFIG guards and a Sanity Pack. Kernelspec: **python3**._


In [120]:
# --- TOP-OF-NOTEBOOK GUARDS / PATHS BOOTSTRAP ---
import os, json, datetime as _dt

# Environment root
if os.path.exists("/kaggle/working"):
    BASE = "/kaggle/working"
elif os.path.exists("/content"):
    BASE = "/content"
else:
    BASE = "/mnt/data"
print("BASE =", BASE)
_p = lambda *p: os.path.join(BASE, *p)

# Single source of truth
CONFIG = {
    "RUN_PIPELINE": True,
    "RUN_UI": True,
    "RUN_MEDS": True,
    "RUN_SYNTH": False,
    "DATA_ROOT": BASE, 
    "EQUIPMENT_STATUS_PATH": _p("equipment_status.csv"),
    "EQUIPMENT_MOVES_LOG_PATH": _p("moves_log.csv"),
    "SOP_REGISTRY_PATH": _p("sop_registry.csv"),
    "QR_OUTPUT_DIR": _p("qrs"),
    "EVENT_LOG_PATH": _p("event_log.jsonl"),
}

# FS prep
os.makedirs(CONFIG["QR_OUTPUT_DIR"], exist_ok=True)
os.makedirs(os.path.dirname(CONFIG["EVENT_LOG_PATH"]), exist_ok=True)

# Logger that always uses current CONFIG (no stale capture)
def _append_event(ev: dict):
    ev = {"ts": _dt.datetime.utcnow().isoformat()+"Z", **(ev or {})}
    with open(CONFIG["EVENT_LOG_PATH"], "a", encoding="utf-8") as f:
        f.write(json.dumps(ev, ensure_ascii=False) + "\n")

print("EVENT_LOG_PATH →", CONFIG["EVENT_LOG_PATH"])


BASE = /kaggle/working
EVENT_LOG_PATH → /kaggle/working/event_log.jsonl


In [121]:
# Freeze CONFIG so later cells can't silently flip flags/paths.
_ALLOWED_CONFIG_KEYS_TO_CHANGE = {"MODEL_BUNDLE_PATH"}  # bundle loader may set this

class _FrozenConfig(dict):
    def __setitem__(self, k, v):
        if k in self and k not in _ALLOWED_CONFIG_KEYS_TO_CHANGE:
            raise RuntimeError(f"CONFIG is frozen; attempted to modify {k}. Keep a single source of truth.")
        super().__setitem__(k, v)
    def update(self, *args, **kwargs):
        if args:
            for k in args[0].keys():
                if k in self and k not in _ALLOWED_CONFIG_KEYS_TO_CHANGE:
                    raise RuntimeError(f"CONFIG is frozen; attempted to modify {k}.")
        for k in list(kwargs.keys()):
            if k in self and k not in _ALLOWED_CONFIG_KEYS_TO_CHANGE:
                raise RuntimeError(f"CONFIG is frozen; attempted to modify {k}.")
        return super().update(*args, **kwargs)

CONFIG = _FrozenConfig(CONFIG)
print("CONFIG frozen. Only allowed later change:", _ALLOWED_CONFIG_KEYS_TO_CHANGE)

CONFIG frozen. Only allowed later change: {'MODEL_BUNDLE_PATH'}


In [126]:
# edtracker self-healing wire (no CONFIG mutation)
from pathlib import Path; import shutil, sys, subprocess, importlib, site
ROOT=Path("/kaggle/working/edtracker_pkg"); PKG=ROOT/"edtracker"; PKG.mkdir(parents=True, exist_ok=True)
pp=ROOT/"pyproject.toml"
if not pp.exists():
    pp.write_text('[build-system]\nrequires=["setuptools","wheel"]\nbuild-backend="setuptools.build_meta"\n\n'
                  '[project]\nname="edtracker"\nversion="0.0.1"\n', encoding="utf-8")
(PKG/"__init__.py").write_text('__all__=["core","ops","ui"]\n', encoding="utf-8")
(PKG/"core").mkdir(exist_ok=True)
(PKG/"core"/"__init__.py").write_text('from .tracker_core import *\n', encoding="utf-8")
tc_dst=PKG/"core"/"tracker_core.py"
if not tc_dst.exists():
    for cand in [Path("/kaggle/input/tracker-core/tracker_core.py"),
                 Path("/kaggle/working/edtracker/core/tracker_core.py")]:
        if cand.exists(): shutil.copy2(cand, tc_dst); break
assert tc_dst.exists(), "tracker_core.py not found"

def copyin(src_root, picks, dst):
    if not src_root: return
    for rel in picks:
        p=src_root/rel
        if p.exists():
            shutil.rmtree(dst, ignore_errors=True); shutil.copytree(p,dst,dirs_exist_ok=True); (dst/"__init__.py").touch(); return
copyin(Path("/kaggle/input/edtracker-ops-pg/edtracker_ops_pkg"), [Path("edtracker/ops"), Path("ops")], PKG/"ops")
copyin(Path("/kaggle/input/edtracker-ui-pkg/edtracker_ui_pkg"), [Path("edtracker/ui"), Path("ui"), Path("edtracker_ui")], PKG/"ui")

subprocess.run(["pip","install","-e",str(ROOT)], check=True); importlib.reload(site)
from edtracker.core.tracker_core import TrackerService
print("✅ edtracker ready →", (PKG/"core"/"tracker_core.py").as_posix())
try: from edtracker.ops import gate_engine; print("   ops:", gate_engine.__module__)
except Exception: print("   ops: (none)")
try: from edtracker.ui import run_ui; print("   ui :", run_ui.__module__)
except Exception: print("   ui : (none)")


Obtaining file:///kaggle/working/edtracker_pkg
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for edtracker (pyproject.toml): started
  Building editable for edtracker (pyproject.toml): finished with status 'done'
  Created wheel for edtracker: filename=edtracker-0.0.1-0.editable-py3-none-any.whl size=2679 sha256=8d1c26a9e71adf3168a0c2e0796aa64b8492f2c5ebec475bad39d558256c578d
  Stored in directory: /tmp/pip-ephem-wheel-cache-x9usjwn0/wheels/19/03/bc/9d8498ee12e3259382e875f69bb7a22e5477304d75e58d20ff
Successfully built edtrac

In [122]:
try:
    from pathlib import Path; import shutil, sys, subprocess, importlib, site
    W=Path("/kaggle/working/edtracker_pkg/edtracker"); W.mkdir(parents=True, exist_ok=True)
    (W/"__init__.py").write_text('__all__=["core","ops","ui"]\n')
    OPS_ROOT=Path("/kaggle/input/edtracker-ops-pg/edtracker_ops_pkg"); UI_ROOT=Path("/kaggle/input/edtracker-ui-pkg/edtracker_ui_pkg")
    pick=lambda c: next((p for p in c if p.is_dir()), None)
    ops_src=pick([OPS_ROOT/"edtracker"/"ops", OPS_ROOT/"ops"])
    ui_src =pick([UI_ROOT/"edtracker"/"ui", UI_ROOT/"ui", UI_ROOT/"edtracker_ui"])
    assert ops_src and ui_src and (W.parent/"pyproject.toml").exists(), "ops/ui or pyproject missing"
    for src,dst in [(ops_src,W/"ops"),(ui_src,W/"ui")]:
        shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src,dst,dirs_exist_ok=True); (dst/"__init__.py").touch()
    subprocess.run(["pip","install","-e",str(W.parent)], check=True); importlib.reload(site); sys.path.insert(0,"/kaggle/working")
    from edtracker.ops import gate_engine; from edtracker.ui import run_ui
    print("wired ops:",gate_engine.__module__,"| ui:",run_ui.__module__)
except Exception as e:
    print("[edtracker wire skipped]", e)


Obtaining file:///kaggle/working/edtracker_pkg
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for edtracker (pyproject.toml): started
  Building editable for edtracker (pyproject.toml): finished with status 'done'
  Created wheel for edtracker: filename=edtracker-0.0.1-0.editable-py3-none-any.whl size=2679 sha256=4d585a99c8cf33e81044ac4070d114d48e686adaaf99104cfe775f596df8df86
  Stored in directory: /tmp/pip-ephem-wheel-cache-oj7twdcl/wheels/19/03/bc/9d8498ee12e3259382e875f69bb7a22e5477304d75e58d20ff
Successfully built edtrac

In [123]:
# ---- edtracker bootstrap (Kaggle/papermill-safe) ----
import sys, importlib, site, pathlib, pkgutil

# 1) Make sure the %pip changes are visible to this process
importlib.reload(site)

# 2) If the package still isn't importable, add your local pkg root to sys.path
PKG_NAME = "edtracker"
PKG_ROOTS = [
    "/kaggle/working/edtracker_pkg",   # your editable source tree
    "/kaggle/working",                 # in case you copied 'edtracker/' directly here
]
if pkgutil.find_loader(PKG_NAME) is None:
    for root in PKG_ROOTS:
        if (pathlib.Path(root) / "edtracker").exists():
            sys.path.insert(0, root)
            break

# 3) Prefer explicit module path to avoid relying on __init__ exports
from edtracker.core.tracker_core import TrackerService
from edtracker.ops import gate_engine

print("✅ edtracker import OK")


✅ edtracker import OK


In [66]:
# --- Load model bundle from attached dataset → set CONFIG path ---
import os, shutil
DATASET = "ed-pipeline-bundle-ui"  # ← replace with your actual dataset slug
SRC = f"/kaggle/input/{DATASET}/ed_phase2_model_thr_patched(1).joblib"
DST = "/kaggle/working/ed_phase2_model_thr_patched(1).joblib"

assert os.path.exists(SRC), f"Not found: {SRC} (check dataset slug/file name)"
if not os.path.exists(DST):
    os.makedirs(os.path.dirname(DST), exist_ok=True)
    shutil.copy2(SRC, DST)

CONFIG["MODEL_BUNDLE_PATH"] = DST
CONFIG["SKIP_MODEL_DISCOVERY"] = True  # prevents fallbacks from overriding this
print("Bundle ready →", CONFIG["MODEL_BUNDLE_PATH"])


Bundle ready → /kaggle/working/ed_phase2_model_thr_patched(1).joblib


In [69]:

# -- Demo meds/ICU/ML flags (non-destructive update of CONFIG) --
CONFIG.setdefault("RUN_MEDS", True)
CONFIG.setdefault("RUN_SYNTH", True)
CONFIG.setdefault("SAVE_SYNTH", True)
CONFIG.setdefault("ALLERGIES_PATH", "/mnt/data/patient_allergies.json")
CONFIG.setdefault("MED_RULES_PATH", "/mnt/data/interaction_rules.json")


'/mnt/data/interaction_rules.json'

In [70]:

# WorkflowState invariant: defined before use; exposes required methods; timers/backlogs/alerts intact
from dataclasses import dataclass, field
from typing import Any, Dict, List
import pandas as pd

@dataclass
class WorkflowState:
    role: str
    state: Dict[str, Any] = field(default_factory=dict)
    timers: Dict[str, float] = field(default_factory=dict)
    backlogs: Dict[str, List[Any]] = field(default_factory=dict)
    alerts: List[str] = field(default_factory=list)

    def touch_now(self, ts: pd.Timestamp):
        # update an example timer
        self.timers["last_touch_epoch"] = float(ts.value) / 1e9

    def feature_dict(self) -> Dict[str, Any]:
        # safe, extendable
        fd = {
            "role": self.role,
            "since_vitals_min": self.state.get("since_vitals_min", 0.0),
            "alerts_count": len(self.alerts),
        }
        # pass through any extra scalar features
        for k,v in self.state.items():
            if isinstance(v,(int,float,str)) and k not in fd:
                fd[k]=v
        return fd

    def update_state_from_event(self, event: Dict[str, Any]):
        # naive: merge event into state; track backlog
        self.state.update(event)
        self.backlogs.setdefault("events", []).append(event)

    def apply_event_log(self, events: List[Dict[str, Any]]):
        for ev in events:
            self.update_state_from_event(ev)


In [71]:

# TinyCritics uses WorkflowState.feature_dict(); cold-start safe (no transform before fit)
import numpy as np

class TinyCritics:
    def __init__(self):
        self._fitted = False

    def fit(self, states, actions, rewards):
        # No-op fit to keep cold-start safe
        self._fitted = True
        return self

    def score(self, state: WorkflowState, actions: List[Dict[str,str]]):
        fd = state.feature_dict()
        n = len(actions)
        # Deterministic, bounded scores in [0,1]
        base = 0.5
        p = np.full(n, base, dtype=float)
        bonuses = np.zeros(n, dtype=float)
        uncertainty = np.full(n, 0.1, dtype=float)
        return p, bonuses, uncertainty


In [72]:
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
from pathlib import Path
import pandas as pd, numpy as np
def _cfg(CONFIG: Any, key: str, default: Any=None) -> Any:
    try: return CONFIG.get(key, default)
    except Exception: return getattr(CONFIG, key, default) if hasattr(CONFIG, key) else default
def _ensure_parent(p: Path): p = Path(p); p.parent.mkdir(parents=True, exist_ok=True)
@dataclass
class EquipmentRecord:
    equip_id: str; name: str=""; location: str=""; status: str=""; last_seen: Optional[str]=None; battery: Optional[float]=None; confidence: Optional[float]=None
    def to_row(self)->Dict[str,Any]: return {"equip_id":self.equip_id,"name":self.name,"location":self.location,"status":self.status,"last_seen":self.last_seen,"battery":self.battery,"confidence":self.confidence}
class EquipmentRepository:
    def __init__(self, status_csv: Path):
        self.status_csv=Path(status_csv); _ensure_parent(self.status_csv)
        if not self.status_csv.exists(): pd.DataFrame(columns=["equip_id","name","location","status","last_seen","battery","confidence"]).to_csv(self.status_csv, index=False)
    def read(self)->pd.DataFrame:
        try: df=pd.read_csv(self.status_csv); 
        except Exception: return pd.DataFrame(columns=["equip_id","name","location","status","last_seen","battery","confidence"])
        if "equip_id" in df.columns: df["equip_id"]=df["equip_id"].astype(str); return df
    def upsert(self, rec: EquipmentRecord)->None:
        df=self.read(); row=pd.DataFrame([rec.to_row()])
        if df.empty: df=row
        else:
            mask=(df["equip_id"].astype(str)==str(rec.equip_id))
            if mask.any(): df.loc[mask,:]=row.values
            else: df=pd.concat([df,row], ignore_index=True)
        df.to_csv(self.status_csv, index=False)
class MovesLogRepository:
    def __init__(self, moves_csv: Path):
        self.moves_csv=Path(moves_csv); _ensure_parent(self.moves_csv)
        if not self.moves_csv.exists(): pd.DataFrame(columns=["equip_id","from","to","ts"]).to_csv(self.moves_csv, index=False)
    def append(self, equip_id:str, loc_from:str, loc_to:str, ts_iso:str)->None:
        row=pd.DataFrame([{"equip_id":equip_id,"from":loc_from,"to":loc_to,"ts":ts_iso}])
        try: prev=pd.read_csv(self.moves_csv) if self.moves_csv.exists() else None; df=pd.concat([prev,row], ignore_index=True) if prev is not None else row
        except Exception: df=row
        df.to_csv(self.moves_csv, index=False)
    def read(self)->pd.DataFrame:
        try: return pd.read_csv(self.moves_csv)
        except Exception: return pd.DataFrame(columns=["equip_id","from","to","ts"])
class SOPRegistry:
    def __init__(self, sop_csv: Path): self.sop_csv=Path(sop_csv); _ensure_parent(self.sop_csv)
    def read(self)->pd.DataFrame:
        if self.sop_csv.exists():
            try:
                df=pd.read_csv(self.sop_csv)
                for col in ["sop_id","title","pdf_path"]:
                    if col not in df.columns: df[col]=""
                return df
            except Exception: pass
        return pd.DataFrame(columns=["sop_id","title","pdf_path","version","status","keywords","checklist","source_url"])
class QRService:
    def __init__(self,out_dir:Path): 
        self.out_dir=Path(out_dir); self.out_dir.mkdir(parents=True, exist_ok=True)
    def make(self,payload:str)->str:
        try:
            import qrcode
            fp=self.out_dir/f"qr_{abs(hash(payload))}.png"
            img=qrcode.make(payload); img.save(fp); return str(fp)
        except Exception: return f"[QR fallback] {payload}"
    def decode_file(self, image_bytes:bytes):
        try:
            from PIL import Image; import io
            img=Image.open(io.BytesIO(image_bytes))
            try:
                from pyzbar.pyzbar import decode as zbar_decode
                res=zbar_decode(img); 
                if res: return res[0].data.decode("utf-8","ignore")
            except Exception: pass
        except Exception: pass
        return None
class TrackerService:
    def __init__(self, equipment_repo:EquipmentRepository, moves_repo:MovesLogRepository, sop_registry:SOPRegistry, qr:QRService, config:Any):
        self.equipment_repo=equipment_repo; self.moves_repo=moves_repo; self.sop_registry=sop_registry; self.qr=qr; self.CONFIG=config
    @classmethod
    def from_config(cls, CONFIG:Any)->"TrackerService":
        return cls(EquipmentRepository(Path(_cfg(CONFIG,"EQUIPMENT_STATUS_PATH"))),
                   MovesLogRepository(Path(_cfg(CONFIG,"EQUIPMENT_MOVES_LOG_PATH"))),
                   SOPRegistry(Path(_cfg(CONFIG,"SOP_REGISTRY_PATH"))),
                   QRService(Path(_cfg(CONFIG,"QR_OUTPUT_DIR"))), CONFIG)
    def equipment_status(self)->pd.DataFrame: return self.equipment_repo.read()
    def log_move(self, equip_id:str, loc_from:str, loc_to:str)->None:
        ts_iso=pd.Timestamp.utcnow().isoformat(); df=self.equipment_repo.read()
        row=df[df["equip_id"].astype(str)==str(equip_id)]; name=row["name"].iloc[0] if not row.empty and "name" in row.columns else ""
        rec=EquipmentRecord(equip_id=equip_id,name=name,location=loc_to,status="moved",last_seen=ts_iso)
        self.equipment_repo.upsert(rec); self.moves_repo.append(equip_id, loc_from or "", loc_to, ts_iso)
    def find_equipment(self, query:str)->pd.DataFrame:
        q=(query or "").strip().lower(); df=self.equipment_repo.read()
        if not q: return df
        def hit(r): return any(q in str(r.get(k,"")).lower() for k in ["equip_id","name","location","status"])
        return df[df.apply(hit, axis=1)]
    def overdue_equipment(self, threshold_minutes:int=120)->pd.DataFrame:
        df=self.equipment_repo.read().copy()
        if df.empty or "last_seen" not in df.columns: return df.iloc[0:0]
        ts=pd.to_datetime(df["last_seen"],errors="coerce",utc=True); age_min=(pd.Timestamp.utcnow().tz_localize("UTC")-ts).dt.total_seconds()/60.0
        df["age_min"]=age_min; return df[age_min>float(threshold_minutes)].sort_values("age_min", ascending=False)
    def movement_stats(self)->Dict[str,pd.DataFrame]:
        log=self.moves_repo.read()
        if log.empty: return {"moves_per_equipment":log,"routes":log}
        per_eq=log.groupby("equip_id").size().reset_index(name="moves").sort_values("moves", ascending=False)
        routes=log.groupby(["from","to"]).size().reset_index(name="count").sort_values("count", ascending=False)
        return {"moves_per_equipment":per_eq,"routes":routes}
    def sop_table(self)->pd.DataFrame: return self.sop_registry.read()
    def search_sop(self, query:str)->pd.DataFrame:
        df=self.sop_registry.read().copy(); q=(query or "").strip().lower()
        if df.empty or not q: return df
        cols=[c for c in ["sop_id","title","keywords","version","status"] if c in df.columns]
        mask=df[cols].astype(str).apply(lambda col: col.str.lower().str.contains(q, na=False)).any(axis=1)
        return df[mask]
    def make_qr(self,payload:str)->str: return self.qr.make(payload)
    def decode_qr_bytes(self, image_bytes:bytes): return self.qr.decode_file(image_bytes)
print("tracker core ready")

tracker core ready


In [73]:
# --- tracker_core alias shim (append-only, no renames) ---
import sys, types
_names = ["TrackerService","QRService","EquipmentRepository","MovesLogRepository","SOPRegistry"]
if "tracker_core" not in sys.modules and all(n in globals() for n in _names):
    _m = types.ModuleType("tracker_core")
    for n in _names:
        setattr(_m, n, globals()[n])
    sys.modules["tracker_core"] = _m
    print("tracker_core alias ready")
else:
    print("tracker_core alias already present or classes missing")


tracker_core alias ready


In [74]:
# --- Patch: make EquipmentRepository.read() robust ---
import pandas as pd, numpy as np

_EQUIP_COLS = ["equip_id","name","location","status","last_seen","battery","confidence"]

def _equiprepo_read_fixed(self):
    try:
        df = pd.read_csv(self.status_csv)
    except Exception:
        # brand-new or unreadable file → start with empty, correct schema
        return pd.DataFrame(columns=_EQUIP_COLS)

    # ensure required columns exist
    for col in _EQUIP_COLS:
        if col not in df.columns:
            df[col] = np.nan

    # normalize types/order and return ALWAYS a DataFrame
    df["equip_id"] = df["equip_id"].astype(str)
    return df[_EQUIP_COLS]

# apply patch without renaming the class
EquipmentRepository.read = _equiprepo_read_fixed
print("Patched EquipmentRepository.read (robust return)")


Patched EquipmentRepository.read (robust return)


In [75]:
from pathlib import Path
from typing import Any, Dict, List
def _slugify(text:str)->str:
    import re; s=re.sub(r"[^a-zA-Z0-9]+","-",text.strip().lower()).strip("-"); return s or "sop"
def refresh_sop_registry(CONFIG: Any, base_url: str="https://sop-notaufnahme.de/sop/")->Dict[str,Any]:
    out_csv=Path(CONFIG["SOP_REGISTRY_PATH"]); pdf_dir=Path(CONFIG["DATA_ROOT"])/"sop_pdfs"; pdf_dir.mkdir(parents=True, exist_ok=True)
    try:
        import requests; from bs4 import BeautifulSoup
    except Exception as e:
        return {"found":0,"saved":0,"errors":1,"error":f"missing libs: {e}"}
    found=saved=errors=0; items=[]
    try:
        r=requests.get(base_url, timeout=15); r.raise_for_status(); soup=BeautifulSoup(r.text,"html.parser")
        links=sorted({a["href"] for a in soup.find_all("a", href=True) if "/product/" in a["href"] and a["href"].startswith("http")})
        for url in links:
            try:
                pr=requests.get(url, timeout=15); pr.raise_for_status(); ps=BeautifulSoup(pr.text,"html.parser")
                ttag=ps.find(["h1","h2"]); title=ttag.get_text(strip=True) if ttag else (ps.find("title").get_text(strip=True) if ps.find("title") else url)
                pdfs=[a["href"] for a in ps.find_all("a", href=True) if a["href"].lower().endswith(".pdf")]
                pdf_url=pdfs[0] if pdfs else None; sop_id=_slugify(title or url.split("/")[-2]); pdf_path=""
                if pdf_url:
                    try:
                        fn=sop_id+".pdf"; outp=pdf_dir/fn
                        with requests.get(pdf_url, stream=True, timeout=30) as dr:
                            dr.raise_for_status()
                            with open(outp,"wb") as f:
                                for chunk in dr.iter_content(8192):
                                    if chunk: f.write(chunk)
                        pdf_path=str(outp); saved+=1
                    except Exception:
                        errors+=1; pdf_path=pdf_url
                items.append({"sop_id":sop_id,"title":title or sop_id,"pdf_path":pdf_path,"version":"","status":"fetched" if pdf_path else "linked","keywords":"","checklist":"","source_url":url})
                found+=1
            except Exception: errors+=1; continue
    except Exception as e:
        return {"found":0,"saved":0,"errors":1,"error":str(e)}
    import pandas as pd
    try:
        if out_csv.exists(): df=pd.read_csv(out_csv)
        else: df=pd.DataFrame(columns=["sop_id","title","pdf_path","version","status","keywords","checklist","source_url"])
        df=df.copy()
        if df.empty: new_df=pd.DataFrame(items)
        else:
            df["sop_id"]=df["sop_id"].astype(str)
            for i in items:
                mask=(df["sop_id"]==str(i["sop_id"]))
                if mask.any():
                    for k,v in i.items():
                        if k in df.columns and (pd.isna(df.loc[mask,k]).all() or str(df.loc[mask,k].iloc[0]).strip()=="" or k in ["pdf_path","status","source_url"]):
                            df.loc[mask,k]=v
                else:
                    df=pd.concat([df, pd.DataFrame([i])], ignore_index=True)
            new_df=df
        new_df.to_csv(out_csv, index=False)
    except Exception as e:
        errors+=1
    return {"found":found,"saved":saved,"errors":errors,"csv":str(out_csv),"dir":str(pdf_dir)}
def load_priority_flows(json_path:str)->Dict[str,Any]:
    import json
    try:
        with open(json_path,"r",encoding="utf-8") as f: return json.load(f)
    except Exception: return {}
print("sop auto ready")

sop auto ready


In [76]:
def rule_hs_tnt(value):
    try: v = float(value)
    except Exception: return 0.50
    if v < 14: return 0.50
    if 14 <= v <= 51: return 0.20
    return 0.50
assert rule_hs_tnt(13.9)==0.50 and rule_hs_tnt(14.0)==0.20 and rule_hs_tnt(51.0)==0.20 and rule_hs_tnt(51.1)==0.50
print("troponin_rules ok")

troponin_rules ok


In [77]:

# Inline SOP surface and optional UI without external modules
from typing import Optional, Dict, Any
import csv, os
from pathlib import Path

def refresh_sop_registry(CONFIG: dict, base_url: Optional[str]) -> Dict[str, Any]:
    """
    Offline-safe: if base_url is provided and fetch works + CSV looks valid, overwrite the file.
    Otherwise, return a summary without raising. No sidecars.
    """
    path = Path(CONFIG["SOP_REGISTRY_PATH"])
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        with path.open("w", newline="") as fp:
            csv.writer(fp).writerow(["id","title","url"])
    if base_url:
        try:
            import requests
            resp = requests.get(base_url, timeout=5)
            resp.raise_for_status()
            text = resp.text.strip()
            rows = [r.split(",") for r in text.splitlines()]
            if rows and len(rows[0])>=3:
                with path.open("w", newline="") as fp:
                    csv.writer(fp).writerows(rows)
                return {"ok": True, "rows": len(rows)-1}
        except Exception as e:
            return {"ok": False, "error": str(e)}
    return {"ok": False, "error": "No base_url or unexpected format"}

# Optional, guarded UI demo (equipment status preview uses the CSV directly)
if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W, pandas as pd
        eq_path = Path(CONFIG["EQUIPMENT_STATUS_PATH"])
        if not eq_path.exists():
            eq_path.write_text("equipment_id,location,last_seen\n")
        btn = W.Button(description="Show equipment status")
        out = W.Output()
        def _on_click(_):
            with out:
                out.clear_output()
                try:
                    df = pd.read_csv(eq_path)
                except Exception:
                    df = pd.DataFrame(columns=["equipment_id","location","last_seen"])
                display(df)
        btn.on_click(_on_click)
        display(W.VBox([btn, out]))
    except Exception as e:
        print("UI unavailable (optional):", e)


In [78]:
# --- UI helpers (split out; no side effects) ---

def run_ui_header_controls():
    import streamlit as st
    c0, c1, c2, c3 = st.columns([2,2,2,2])
    with c0:
        thresh = st.number_input("Overdue threshold (min)", min_value=5, max_value=720, value=120, step=5)
    with c1:
        # NEW: adjustable vitals threshold (replaces hard-coded 120)
        vitals_thresh = st.number_input("Vitals overdue (min)", min_value=5, max_value=720, value=120, step=5)
    with c2:
        if st.button("Refresh"):
            st.experimental_rerun()
    return thresh, vitals_thresh

def run_ui_equipment_panel(tracker, overdue_thresh_min: float):
    import streamlit as st, pandas as pd
    st.header("Equipment")
    eq_df = tracker.equipment_status()
    s1, s2 = st.columns([2,1])
    with s1:
        q = st.text_input("Find equipment (ID / name / location / status)", "")
        filt = tracker.find_equipment(q) if q else eq_df
        st.dataframe(filt, use_container_width=True, height=260)
    with s2:
        overdue = tracker.overdue_equipment(int(overdue_thresh_min))
        st.subheader("Overdue")
        if overdue.empty:
            st.write("None")
        else:
            st.dataframe(overdue[["equip_id","name","location","last_seen","age_min"]],
                         use_container_width=True, height=200)
    st.markdown("**Update location / log move**")
    mc1, mc2, mc3, mc4 = st.columns([2,2,2,1])
    with mc1:
        sel_id = st.selectbox("Equipment ID", [""] + sorted(list(eq_df.get("equip_id", []))))
    with mc2:
        loc_from = st.text_input("From", "")
    with mc3:
        loc_to = st.text_input("To", "")
    with mc4:
        if st.button("Log move") and sel_id and loc_to:
            tracker.log_move(sel_id, loc_from, loc_to)
            st.success(f"Move logged: {sel_id} → {loc_to}")

def run_ui_qr_panel(tracker):
    import streamlit as st
    st.header("QR")
    qr_col1, qr_col2 = st.columns([2,2])
    with qr_col1:
        qr_txt = st.text_input("QR payload to generate", "")
        if st.button("Generate QR") and qr_txt:
            path = tracker.make_qr(qr_txt)
            st.write("QR saved to:", path)
    with qr_col2:
        st.write("Scan and update location")
        f = st.file_uploader("Upload QR image", type=["png","jpg","jpeg","webp"])
        manual_payload = st.text_input("Manual payload (fallback if decoding fails)", "")
        new_loc = st.text_input("New location (after scan)", "")
        if st.button("Scan & Update"):
            equip_payload = None
            if f is not None:
                equip_payload = tracker.decode_qr_bytes(f.read())
            if not equip_payload and manual_payload:
                equip_payload = manual_payload
            if equip_payload and new_loc:
                equip_id = equip_payload
                if "id=" in equip_payload:
                    try:
                        equip_id = equip_payload.split("id=",1)[1].split("&",1)[0]
                    except Exception:
                        equip_id = equip_payload
                tracker.log_move(str(equip_id), "", new_loc)
                st.success(f"Updated via payload. {equip_id} → {new_loc}")
            elif not new_loc:
                st.error("Provide a new location.")
            else:
                st.error("No QR payload detected (image or manual).")

def run_ui_sop_panel():
    import streamlit as st
    with st.expander("SOP auto-pull and flows", expanded=False):
        if st.button("Refresh SOPs from sop-notaufnahme.de"):
            # assumes refresh_sop_registry(CONFIG, ...) exists in scope
            res = refresh_sop_registry(CONFIG, base_url="https://sop-notaufnahme.de/sop/")
            st.write(res)
        flows = load_priority_flows("/mnt/data/priority_flows.json")
        if flows:
            keys = sorted(list(flows.keys()))
            pickf = st.selectbox("Show flow", [""] + keys)
            if pickf:
                flow = flows[pickf]
                st.subheader(flow.get("title", pickf))
                nodes = flow.get("nodes", []); edges = flow.get("edges", [])
                st.write("Nodes:", ", ".join([n.get("label", n.get("id","")) for n in nodes]))
                try:
                    import matplotlib.pyplot as plt
                    fig = plt.figure()
                    pos = {n["id"]:(i, 0) for i,n in enumerate(nodes)}
                    for n in nodes:
                        x,y = pos[n["id"]]; plt.scatter([x],[y])
                        plt.text(x,y+0.05,n.get("label", n["id"]), ha="center", rotation=45)
                    for a,b in edges:
                        xa,ya = pos.get(a,(0,0)); xb,yb = pos.get(b,(0,0))
                        plt.plot([xa,xb],[ya,yb])
                    plt.axis("off"); plt.title(flow.get("title", pickf))
                    st.pyplot(fig)
                except Exception:
                    st.info("Graph display unavailable; showing list instead.")
                    st.write(edges)

def run_ui_actions_and_critic(get_state, get_actions, critic, vitals_thresh_min: float):
    import streamlit as st, pandas as pd, numpy as np
    st.header("SOPs")
    # assumes `tracker` in wrapper handles search UI; keeping your SOP table UI below
    # (if you want SOP search to stay here, we can move it — keeping behavior same is safer)
    st.header("Actions & Critic")
    state = get_state()
    if hasattr(state, "feature_dict"):
        feats = state.feature_dict()
        since_v = feats.get("since_vitals_min", None)
        if since_v is not None:
            # NEW: use adjustable threshold instead of hard-coded 120
            if since_v > float(vitals_thresh_min):
                st.error(f"Lingering patient: since_vitals_min={since_v:.0f} > {int(vitals_thresh_min)}")
            else:
                st.success(f"Vitals recently checked: {since_v:.0f} min (≤ {int(vitals_thresh_min)})")
    if st.button("Mark vitals now") and hasattr(state, "touch_now"):
        state.touch_now(pd.Timestamp.utcnow())
        st.success("Vitals timestamp updated.")
    actions = get_actions(state)
    if not actions:
        st.info("No actions available.")
        return
    p, benefit, burden = critic.score(state, actions)
    view = pd.DataFrame({
        "id":[a.get("id") for a in actions],
        "label":[a.get("label") for a in actions],
        "p_accept":np.round(p,3),
        "benefit":np.round(benefit,3),
        "burden":np.round(burden,3)
    }).sort_values(["p_accept","benefit"], ascending=[False, False])
    st.dataframe(view, use_container_width=True, height=240)

def run_ui_movement_analytics(tracker):
    import streamlit as st
    st.header("Equipment Movement Analytics")
    stats = tracker.movement_stats()
    per_eq = stats["moves_per_equipment"]; routes = stats["routes"]
    if per_eq.empty:
        st.info("No movement data yet.")
    else:
        st.subheader("Moves per equipment")
        st.dataframe(per_eq, use_container_width=True, height=240)
        try:
            import matplotlib.pyplot as plt
            fig = plt.figure()
            x = per_eq["equip_id"].astype(str).tolist()
            y = per_eq["moves"].tolist()
            plt.bar(x,y); plt.xticks(rotation=45, ha="right"); plt.title("Moves per Equipment")
            st.pyplot(fig)
        except Exception:
            pass
        st.subheader("Top routes")
        st.dataframe(routes, use_container_width=True, height=200)


In [79]:
# --- Wrapper: calls helpers; same signature as before ---
def run_ui(tracker, get_state, get_actions, critic):
    import streamlit as st, pandas as pd, numpy as np
    st.set_page_config(page_title="ED Tracker — Full", layout="wide")
    st.title("ED Tracker — Core Ops (Full)")

    # Header controls (now returns both thresholds)
    overdue_thresh, vitals_thresh = run_ui_header_controls()

    # Panels
    run_ui_equipment_panel(tracker, overdue_thresh)
    run_ui_qr_panel(tracker)
    run_ui_sop_panel()

    # SOP search table (kept here to mirror your original flow)
    sop_q = st.text_input("Search SOPs (id/title/keywords)", "")
    sop_hits = tracker.search_sop(sop_q)
    if sop_hits.empty:
        st.info("No SOPs found.")
    else:
        st.dataframe(sop_hits[["sop_id","title","version","status"]], use_container_width=True, height=220)
        pick = st.selectbox("Open SOP", [""] + sop_hits["sop_id"].astype(str).tolist())
        if pick:
            row = sop_hits[sop_hits["sop_id"].astype(str)==pick].iloc[0]
            pdf = row.get("pdf_path","")
            if pdf:
                st.write("PDF path:", pdf)
            if "checklist" in sop_hits.columns and isinstance(row.get("checklist", None), str) and row["checklist"].strip():
                st.subheader("Checklist")
                steps = [s.strip() for s in row["checklist"].split("|") if s.strip()]
                completed = []
                for i, step in enumerate(steps, 1):
                    if st.checkbox(f"{i}. {step}", key=f"sop_{pick}_{i}"):
                        completed.append(i)
                st.caption(f"Completed {len(completed)}/{len(steps)} steps")

    # Actions, critic, and analytics (uses adjustable vitals threshold)
    run_ui_actions_and_critic(get_state, get_actions, critic, vitals_thresh)
    run_ui_movement_analytics(tracker)
    print("ui ready")


In [80]:

# Inline UI trigger (no sidecars)
if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W
        btn = W.Button(description="Run ML → OPS (inline)", button_style="primary")
        out = W.Output()
        def _go(_):
            with out:
                out.clear_output()
                print("Running…")
                try:
                    test, probs_te, tau, id_col, src = run_user_pipeline()
                    res = ml_to_ops_emit(test, probs_te, tau, id_col, src)
                    print("[OK] Emitted:", res)
                except Exception as e:
                    print("[ERROR]", e)
        btn.on_click(_go)
        display(W.VBox([btn, out]))
    except Exception as e:
        print("UI unavailable:", e)
else:
    print("UI panel deferred… set CONFIG['RUN_UI']=True.")


In [81]:
# === MLP → OPS advice-only bridge (merged, guarded, contract-compliant) ===
# Runs only when CONFIG["RUN_PIPELINE"] is True. Only CONFIG["MODEL_BUNDLE_PATH"] may be set.
import os, glob, json
from pathlib import Path
import numpy as np, pandas as pd

if CONFIG.get("RUN_PIPELINE", False):
    # --- 1) Locate model bundle (prefer explicit path, then dataset, then glob) ---
    MODEL_HINT = CONFIG.get("MODEL_BUNDLE_PATH")
    SEARCH_DIRS = [
        (MODEL_HINT if MODEL_HINT else None),
        "/kaggle/input/ed-pipeline-bundle-ui",
        "/kaggle/working",
        "/mnt/data",
        "/content",
    ]
    candidates = []
    for root in [d for d in SEARCH_DIRS if d]:
        p = Path(root)
        if p.is_file() and str(p).endswith((".joblib",".pkl")):
            candidates.append(str(p))
        elif p.exists():
            candidates += [str(x) for x in p.rglob("ed_phase2_model*_patched(1).joblib")]
            candidates += [str(x) for x in p.rglob("*.joblib")]
            candidates += [str(x) for x in p.rglob("*.pkl")]
    BUNDLE = next((p for p in candidates if Path(p).exists()), None)
    if not BUNDLE:
        raise FileNotFoundError("Model bundle not found. Set CONFIG['MODEL_BUNDLE_PATH'] or add dataset.")

    # Only allowed mutation per contract
    CONFIG["MODEL_BUNDLE_PATH"] = BUNDLE

    # --- 2) Load bundle (no side effects) ---
    from joblib import load
    b = load(BUNDLE)
    pipe = b.get("pipeline") or b.get("model")
    cal  = b.get("calibrator", None)  # may be None
    THR  = float(b.get("threshold", 0.5))
    FEAT = b.get("features")          # list or None
    if pipe is None:
        raise RuntimeError("pipeline missing in joblib bundle")

    # --- 3) Build minimal legacy feature frame from MIMIC-IV-ED demo ---
    MIMIC_BASE = Path("/kaggle/input/mimic-iv-demo-v2-2")
    ed  = pd.read_csv(MIMIC_BASE/"edstays.csv")
    tri = pd.read_csv(MIMIC_BASE/"triage.csv")

    # times
    for c in ("intime","outtime"):
        if c in ed.columns: ed[c] = pd.to_datetime(ed[c])

    # triage shim → legacy names
    keep = [c for c in ["stay_id","acuity","heartrate","sbp","dbp","chiefcomplaint"] if c in tri.columns]
    tri_small = tri[keep].drop_duplicates("stay_id")
    tri_small["HF"] = tri_small.get("heartrate")
    tri_small["MAP"] = np.nan
    if {"sbp","dbp"}.issubset(tri_small.columns):
        tri_small["MAP"] = tri_small["dbp"] + (tri_small["sbp"] - tri_small["dbp"]) / 3.0
    tri_small["Triage"] = pd.to_numeric(tri_small.get("acuity"), errors="coerce")

    df = ed.merge(tri_small[["stay_id","HF","MAP","Triage","chiefcomplaint"]], on="stay_id", how="left")

    # construct required feature order
    feat_names = FEAT or ['Tag','t_min','Triage','Leitsymptom','HF','MAP','ICU_Kap','Kap_veraltet',
                          't_norm','hat_Labor','Labor_ausstehend','hat_Roentgen','Roentgen_ausstehend',
                          'hat_CT','CT_ausstehend','naechste_Aktion']
    X = pd.DataFrame(index=df.index, columns=feat_names, dtype="float32")

    # populate available numerics
    if "HF" in X.columns:     X["HF"] = df["HF"].astype("float32")
    if "MAP" in X.columns:    X["MAP"] = df["MAP"].astype("float32")
    if "Triage" in X.columns: X["Triage"] = df["Triage"].astype("float32")

    # simple time features
    if "Tag" in X.columns:
        X["Tag"] = (ed["intime"].dt.hour if "intime" in ed.columns else 0).fillna(0).astype("float32")
    if "t_min"  in X.columns: X["t_min"]  = 0.0
    if "t_norm" in X.columns: X["t_norm"] = 0.0

    # defaults for others
    for c in ["ICU_Kap","Kap_veraltet","hat_Labor","Labor_ausstehend","hat_Roentgen",
              "Roentgen_ausstehend","hat_CT","CT_ausstehend","naechste_Aktion"]:
        if c in X.columns: X[c] = 0.0
    if "Leitsymptom" in X.columns:
        # leave NaN to let the pipeline imputer handle; avoids free-text leakage
        X["Leitsymptom"] = np.nan

    X = X.apply(pd.to_numeric, errors="coerce").astype("float32")

    # --- 4) Score (advice-only), with calibrator fallback and tiny smoke ---
    def _score_proba(_X: pd.DataFrame) -> np.ndarray:
        p = pipe.predict_proba(_X)[:, 1]
        # support sklearn IsotonicRegression (predict) and calibrated wrappers (transform not guaranteed)
        if cal is None:
            return p
        return cal.predict(p) if hasattr(cal, "predict") else (cal.transform(p) if hasattr(cal, "transform") else p)

    p_raw = pipe.predict_proba(X)[:, 1]
    try:
        p_cal = _score_proba(X)
    except Exception:
        p_cal = p_raw  # safe fallback
    advice = (p_cal >= THR) if np.isfinite(THR) else np.zeros_like(p_cal, dtype=bool)

    out = pd.DataFrame({
        "stay_id": df["stay_id"],
        "p_gatepos_mlp_raw": p_raw,
        "p_gatepos_mlp_cal": p_cal,
        "advice_gatepos_bool": advice,   # advice-only; NOT wired to gates
    })
    out_path = "/kaggle/working/advice_gatepos_mlp.csv"
    out.to_csv(out_path, index=False)

    # --- 5) Join ops panel if present (non-blocking advice card) ---
    panel_path = "/kaggle/working/advice_ops_panel.csv"
    if Path(panel_path).exists():
        panel = pd.read_csv(panel_path)
        panel = panel.merge(out[["stay_id","p_gatepos_mlp_cal","advice_gatepos_bool"]],
                            on="stay_id", how="outer")
        panel.to_csv(panel_path, index=False)

    # --- 6) Tiny smoke (file exists; lengths match; probs in [0,1]) ---
    assert Path(out_path).exists(), "advice CSV not written"
    assert len(out) == len(df), "row count mismatch"
    assert np.isfinite(out["p_gatepos_mlp_raw"]).all(), "NaNs in raw probs"
    ok_cal = (out["p_gatepos_mlp_cal"] >= 0 - 1e-9) & (out["p_gatepos_mlp_cal"] <= 1 + 1e-9)
    assert ok_cal.all(), "calibrated probs out of [0,1]"

    print(f"ML_BRIDGE_SMOKE_OK | model={Path(BUNDLE).name} | rows={len(out)} | thr={THR}")
else:
    print("ML bridge skipped (CONFIG['RUN_PIPELINE'] is False)")


ML_BRIDGE_SMOKE_OK | model=ed_phase2_model_thr_patched(1).joblib | rows=222 | thr=0.9988444286248084


/usr/local/lib/python3.11/dist-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator SimpleImputer from version 1.1.3 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator StandardScaler from version 1.1.3 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator LabelBinarizer from version 1.1.3 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For mor

In [82]:
# After the bridge cell
def score_and_log(row: dict):
    """row must have the 16 feature keys in the bundle (FEAT)."""
    res = predict_one(row)  # uses THR/FEAT from the bridge
    try:
        _append_event({"type":"ml_score", "res": res, "keys": list(row.keys())})
    except Exception:
        pass
    print(f"ML risk p={res['p']:.3f} → {'ALERT' if res['y'] else 'ok'} (thr={res['thr']:.3f})")
    return res


In [83]:
# ICU availability panel — replaces "actionable vs blocked" with explicit next-bed ETAs.
# No sidecars; all inline; guarded by RUN_UI.
from pathlib import Path
from datetime import datetime, timezone
import json

# Pre-seeded UKE ICUs (public info): names + capacities
UKE_UNITS = [
    {"name": "1A Neurochirurgische Intensivstation", "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1B Neurologische Intensivstation",    "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1C Interdisziplinäre Intensivstation","capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1D Interdisziplinäre Intensivstation","capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1E Interdisziplinäre Intensivstation","capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1F Operative Intensivstation",        "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1G Internistische Intensivstation",   "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "H1b Kardiologische Intensivstation",  "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "H1b Kardiochirurgische Intensivstation","capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "H2b Intensivstation Gefäß- und Herzmedizin", "capacity": 8, "occupied": 8, "discharge_eta_minutes": []},
]

def _format_eta(mins: int) -> str:
    if mins is None: return "unknown"
    if mins <= 0: return "now"
    h, m = divmod(int(mins), 60)
    return f"{m} min" if h == 0 else (f"{h} hr" if m == 0 else f"{h} hr {m} min")

def _load_icu_status(path="/mnt/data/icu_status.json"):
    p = Path(path)
    if p.exists():
        try:
            js = json.loads(p.read_text())
            if isinstance(js, dict) and js.get("units"):
                return js
        except Exception:
            pass
    # default to UKE units when nothing saved
    return {"units": list(UKE_UNITS), "timestamp": datetime.now(timezone.utc).isoformat()}

def _save_icu_status(js, path="/mnt/data/icu_status.json"):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(js, ensure_ascii=False, indent=2))

def _compute_next_bed_eta(unit):
    cap = int(unit.get("capacity", 0) or 0)
    occ = int(unit.get("occupied", 0) or 0)
    etas = [int(x) for x in (unit.get("discharge_eta_minutes") or []) if str(x).strip().isdigit()]
    if occ < cap: return 0
    return min(etas) if etas else None

if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W
        import pandas as pd

        state = _load_icu_status()
        units = state["units"]

        # UI widgets
        dd = W.Dropdown(options=[u.get("name","(unnamed)") for u in units] or ["(add a unit)"], description="Unit")
        name = W.Text(description="Name", placeholder="ICU-North")
        cap  = W.IntText(description="Capacity", value=12)
        occ  = W.IntText(description="Occupied", value=12)
        eta  = W.Text(description="ETAs (min)", placeholder="e.g. 30, 90, 180")

        add_btn = W.Button(description="Add/Update unit")
        calc_btn = W.Button(description="Estimate ETAs")
        save_btn = W.Button(description="Save status")
        out = W.Output()

        def _refresh_dropdown():
            dd.options = [u.get("name","(unnamed)") for u in units] or ["(add a unit)"]

        def _load_into_form(idx=0):
            if not units:
                name.value=""; cap.value=12; occ.value=12; eta.value=""; return
            u = units[idx]
            name.value = str(u.get("name",""))
            cap.value = int(u.get("capacity", 0) or 0)
            occ.value = int(u.get("occupied", 0) or 0)
            seq = u.get("discharge_eta_minutes") or []
            eta.value = ", ".join(str(int(x)) for x in seq)

        def _parse_eta(txt: str):
            out = []
            for chunk in txt.split(","):
                chunk = chunk.strip()
                if chunk:
                    try: out.append(int(float(chunk)))
                    except: pass
            return out

        def on_dd_change(change):
            if change["name"]=="value" and units:
                _load_into_form(dd.options.index(change["new"]))
        dd.observe(on_dd_change)

        def on_add(_):
            # no 'nonlocal' needed: we mutate the existing list
            u = {
                "name": name.value.strip() or f"ICU-{len(units)+1}",
                "capacity": int(cap.value or 0),
                "occupied": int(occ.value or 0),
                "discharge_eta_minutes": _parse_eta(eta.value),
            }
            names = [x.get("name","") for x in units]
            if u["name"] in names:
                units[names.index(u["name"])] = u
            else:
                units.append(u)
            _refresh_dropdown()
            dd.value = u["name"]
            with out:
                print(f"Saved unit '{u['name']}'")

        def on_calc(_):
            rows = []
            for u in units:
                eta_min = _compute_next_bed_eta(u)
                rows.append({
                    "ICU": u.get("name",""),
                    "capacity": int(u.get("capacity",0) or 0),
                    "occupied": int(u.get("occupied",0) or 0),
                    "next_bed_in": _format_eta(eta_min),
                })
            df = pd.DataFrame(rows) if rows else pd.DataFrame(columns=["ICU","capacity","occupied","next_bed_in"])
            with out:
                out.clear_output()
                if df.empty:
                    print("No units yet. Add a unit above.")
                else:
                    display(df.style.hide(axis='index'))
                    # Natural-language earliest
                    mins = [(r["ICU"], _compute_next_bed_eta(u)) for r,u in zip(rows, units)]
                    mins = [(n,m) for n,m in mins if m is not None]
                    if mins:
                        name_min, m = sorted(mins, key=lambda x: x[1])[0]
                        print(f"\nNext bed available on {name_min} in {_format_eta(m)}")
                    else:
                        print("\nNext bed availability: unknown (provide ETAs or reduce occupied < capacity).")

        def on_save(_):
            state["units"] = units
            state["timestamp"] = datetime.now(timezone.utc).isoformat()
            _save_icu_status(state)
            with out:
                print("Saved to /mnt/data/icu_status.json")

        add_btn.on_click(on_add)
        calc_btn.on_click(on_calc)
        save_btn.on_click(on_save)

        # Initial load
        _refresh_dropdown()
        if units:
            dd.value = dd.options[0]
            _load_into_form(0)

        display(W.VBox([
            W.HTML("<b>ICU next-bed availability</b>"),
            dd,
            W.HBox([name, cap, occ]),
            eta,
            W.HBox([add_btn, calc_btn, save_btn]),
            out
        ]))

    except Exception as e:
        print("ICU UI unavailable:", e)
else:
    print("ICU UI deferred… set CONFIG['RUN_UI']=True.")


In [84]:
# === Synth helpers (timestamps + case generator) ===
# Pure helpers. No side effects. Uses only fields consumed by gate_engine/ops_intel/troponin_policy.

import random, json
from datetime import datetime, timezone

def _ts() -> str:
    return datetime.now(timezone.utc).isoformat()

def synth_patient(seed=None) -> dict:
    """
    Generate one realistic ED case.
    SBP-centric vitals, optional VBG, abdomen guardrails, and troponin timing for due/overdue.
    """
    rng = random.Random(seed if seed is not None else random.randint(0, 2**31-1))
    case = {}

    # Triage & vitals (SBP, not MAP)
    case["Triage"] = rng.choices(["Green","Yellow","Orange","Red"], weights=[45,30,20,5], k=1)[0]
    case["SBP"]    = max(70, int(rng.gauss(115, 20)))              # mmHg
    case["HR"]     = max(40, int(rng.gauss(92, 20)))
    case["SpO2"]   = max(70, int(rng.gauss(95, 4)))
    case["GCS"]    = rng.choices([15, 14, 13], weights=[90,7,3])[0]

    # Chief complaint
    cc = rng.choices(
        ["Brustschmerz","Bauchschmerz","Dyspnoe","Synkope","Fieber","Schwindel"],
        weights=[18,22,18,10,20,12], k=1
    )[0]
    case["Leitsymptom"] = cc

    # VBG in ~40%: mostly normal, but inject tails to exercise gates/alerts
    if rng.random() < 0.4:
        case["vbg_pH"]        = round(rng.gauss(7.40, 0.06), 2)
        case["vbg_pco2_mmHg"] = round(max(30, rng.gauss(42, 12)), 1)
        case["vbg_HCO3"]      = round(max(8, rng.gauss(24, 5)), 1)
        case["vbg_lactate"]   = round(max(0.5, rng.gauss(1.8, 0.9)), 1)
        case["vbg_Na"]        = round(rng.gauss(139, 6), 1)
        case["vbg_Cl"]        = round(rng.gauss(103, 6), 1)
        case["vbg_K"]         = round(rng.gauss(4.1, 0.7), 1)

        # Rare criticals (aligned with your ops_intel thresholds)
        if rng.random() < 0.05: case["vbg_pH"]      = 7.56   # alkalosis critical (>7.55)
        if rng.random() < 0.05: case["vbg_K"]       = 6.6    # hyperK critical (>=6.5)
        if rng.random() < 0.05: case["vbg_lactate"] = 4.3    # lactate critical (>=4.0)
        if rng.random() < 0.03: case["vbg_Na"]      = 124    # hyponatraemia critical (<=125)

    # Abdomen guardrails (force unmet requirements to trigger gates)
    if "bauch" in cc.lower() or "abd" in cc.lower():
        case["US_done"] = False; case["CT_done"] = False
        case["labs_cycles_done"] = 1

    # Troponin: enable due/overdue logic; we’re not doing assay inference here
    if cc == "Brustschmerz" and rng.random() < 0.7:
        case["troponin0_ngL"]  = rng.choice([8, 10, 14, 20, 55])  # includes potential rule-in t0
        case["since_trop1_min"]= rng.choice([30, 65, 80])         # to trigger DUE/OVERDUE
        case["admit_decision"] = rng.random() < 0.5

    return case


In [85]:
# === Gate-only stream emitter (no MLP, writes JSONL) ===
# Pure function; no side effects unless you call it.

from collections import Counter
from pathlib import Path
from typing import Dict, Any

def emit_events(n: int = 50, seed: int = 42, sleep_s: float = 0.0,
                CONFIG: Dict[str, Any] = None, EVENT_LOG_PATH: str | None = None):
    assert CONFIG is not None, "CONFIG required"
    log_path = EVENT_LOG_PATH or CONFIG.get("EVENT_LOG_PATH", "/kaggle/working/event_log.jsonl")
    Path(log_path).parent.mkdir(parents=True, exist_ok=True)
    Path(log_path).touch(exist_ok=True)

    gate_counts, prio_hist, actions = Counter(), Counter(), Counter()

    for i in range(n):
        fd   = synth_patient(seed + i)
        pack = gate_engine(fd, CONFIG)  # uses your troponin_policy + ops_intel

        for g in pack.get("gates", []): gate_counts[g] += 1
        prio_hist[pack.get("priority", 0)] += 1
        if pack.get("next_action"): actions[pack["next_action"]] += 1

        rec = {
            "ts": _ts(),
            "case_id": f"synth-{seed}-{i}",
            "fd": fd,
            "gates": pack.get("gates", []),
            "priority": pack.get("priority", 0),
            "next_action": pack.get("next_action", ""),
            "ttl": pack.get("ttl", {}),
            "explain": pack.get("explain", []),
        }
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

        if sleep_s:  # optional pacing
            import time; time.sleep(float(sleep_s))

    # Optional tabular summary (pandas if available)
    try:
        import pandas as pd
        df = (pd.DataFrame({"gate": list(gate_counts.keys()), "count": list(gate_counts.values())})
                .sort_values("count", ascending=False).reset_index(drop=True))
        print("Gate frequency (top 10):"); display(df.head(10))
    except Exception:
        print("Gate frequency (top 10):", gate_counts.most_common(10))
    print("Priority histogram:", dict(prio_hist))
    print("Top actions:", actions.most_common(5))

    return {"gate_counts": gate_counts, "prio_hist": prio_hist, "actions": actions, "log_path": log_path}


In [86]:
# === ICU capacity gate-pack (tiny & pure; reuses ICU_PREFS if present) ===
from typing import Dict, Any, List

def _best_relevant_icu_capacity(fd: Dict[str, Any], ICU_PREFS: Dict[str, List[str]]) -> float:
    cond = fd.get("suspected_condition")
    keys = ICU_PREFS.get(cond, [])
    vals = []
    for k in keys:
        try: vals.append(float(fd.get(k, 0.0)))
        except Exception: pass
    return max(vals) if vals else 0.0

def capacity_gatepack(fd: Dict[str, Any], CONFIG: Dict[str, Any]) -> Dict[str, Any]:
    # Prefer your panel's ICU_PREFS if it exists; else fallback
    ICU_PREFS = CONFIG.get("ICU_PREFS", globals().get("ICU_PREFS", {
        "cardiac":         ["cap_cardio_icu","cap_cardio_surgery_icu","cap_vascular_cardiac_icu"],
        "respiratory":     ["cap_internal_medicine_icu","cap_interdis_stage1","cap_interdis_stage2","cap_interdis_stage3"],
        "neurological":    ["cap_neurological_icu","cap_neurochirurgical_icu","cap_interdis_stage1"],
        "infection":       ["cap_internal_medicine_icu","cap_interdis_stage2"],
        "trauma":          ["cap_surgical_icu","cap_interdis_stage3"],
        "oncology":        ["cap_internal_medicine_icu","cap_interdis_stage2"],
        "gastrointestinal":["cap_surgical_icu","cap_internal_medicine_icu"],
    }))
    TH_OK    = float(CONFIG.get("TH_ICU_CAP_OK",    0.25))  # ≥ OK
    TH_TIGHT = float(CONFIG.get("TH_ICU_CAP_TIGHT", 0.15))  # < tight
    SLA_ESC  = int(CONFIG.get("SLA_ADMIT_ESCALATE_MIN", 60))

    triage_txt = str(fd.get("Triage","")).title()
    urgent = (int(fd.get("ems_triage_code", 3)) <= 2) or (triage_txt in ("Red","Orange"))

    cap_rel    = _best_relevant_icu_capacity(fd, ICU_PREFS)  # 0..1
    bottleneck = 1.0 - cap_rel

    gates: List[str] = []; explain: List[str] = []; ttl: Dict[str,int] = {}
    prio, action = 0, ""

    if cap_rel <= 0.0:
        gates += ["ICU_BLOCKED"]; explain += ["Relevant ICU capacity = 0%"]
        prio = max(prio, 5); action = action or "No ICU capacity — board in ED, activate bed manager now."
    elif cap_rel < TH_TIGHT:
        gates += ["ICU_TIGHT"]; explain += [f"Relevant ICU capacity < {int(TH_TIGHT*100)}%"]
        prio = max(prio, 5 if urgent else 4); action = action or "Tight ICU capacity — escalate to bed manager."
    elif cap_rel >= TH_OK:
        gates += ["ICU_OK"]; explain += [f"Relevant ICU capacity ≥ {int(TH_OK*100)}%"]
        prio = max(prio, 3 if urgent else 2); action = action or "Proceed with ICU handoff planning."

    if bottleneck >= 0.70:
        gates += ["ED_BOARDING_RISK"]; explain += ["High boarding risk from ICU bottleneck"]
        prio = max(prio, 4)

    if urgent and cap_rel >= TH_OK:
        gates += ["ROUTE_PLAN_EMS_TO_ICU"]; explain += ["Urgent + capacity OK → EMS→ICU"]
    else:
        gates += ["ROUTE_PLAN_ED_TO_ICU"];  explain += ["Default or constrained → EMS→ED→ICU"]

    if any(g in ("ICU_BLOCKED","ICU_TIGHT") for g in gates):
        for g in ("ALERT_NURSE","ALERT_DOC"):
            if g not in gates: gates.append(g); ttl[g] = 300
        if ("ICU_BLOCKED" in gates) or (urgent and "ICU_TIGHT" in gates):
            if "ALERT_ATTENDING" not in gates: gates.append("ALERT_ATTENDING"); ttl["ALERT_ATTENDING"] = 300

    if str(fd.get("admit_decision","")).upper() == "ICU" and any(g in ("ICU_BLOCKED","ICU_TIGHT") for g in gates):
        gates.append("TRANSFER_BLOCK_CAPACITY"); explain.append(f"Recheck capacity q{SLA_ESC}min until cleared")
        ttl["TRANSFER_BLOCK_CAPACITY"] = SLA_ESC * 60

    # de-dup preserve order
    seen=set(); gates=[g for g in gates if (g not in seen and not seen.add(g))]
    return {"gates": gates, "priority": prio, "next_action": action, "explain": explain, "ttl": ttl}


In [87]:
# === Guarded execution (set CONFIG['RUN_SYNTH']=True to run) ===
# Interface constraint: no side effects unless RUN_SYNTH is True.

from pathlib import Path

if CONFIG.get("RUN_SYNTH", False):
    out = emit_events(
        n        = int(CONFIG.get("SYNTH_N", 100)),
        seed     = int(CONFIG.get("SYNTH_SEED", 42)),
        sleep_s  = float(CONFIG.get("SYNTH_SLEEP_S", 0.0)),
        CONFIG   = CONFIG,
        EVENT_LOG_PATH = CONFIG.get("EVENT_LOG_PATH", "/kaggle/working/event_log.jsonl"),
    )
    # Minimal smoke: file exists and not empty
    lp = out["log_path"]
    assert Path(lp).exists() and Path(lp).stat().st_size > 0, "Event log not written"
    print("SYNTH_STREAM_OK →", lp)
else:
    print("Deferred… set CONFIG['RUN_SYNTH']=True to generate + run synthetic pipeline.")


Deferred… set CONFIG['RUN_SYNTH']=True to generate + run synthetic pipeline.


In [88]:
# Extend CONFIG for meds integration (idempotent, Kaggle-safe)
CONFIG.setdefault("MED_RULES_PATH", _p("interaction_rules.json"))
CONFIG.setdefault("ALLERGIES_PATH", _p("patient_allergies.json"))

from pathlib import Path
import json, os

# Ensure files exist and are valid JSON arrays (no seeding of drug examples here)
for k in ("MED_RULES_PATH","ALLERGIES_PATH"):
    p = Path(CONFIG[k]); p.parent.mkdir(parents=True, exist_ok=True)
    if not p.exists() or os.path.getsize(p) == 0:
        p.write_text("[]", encoding="utf-8")
    try:
        json.loads(p.read_text(encoding="utf-8") or "[]")
    except Exception:
        p.write_text("[]", encoding="utf-8")
print("Meds/allergy paths →", CONFIG["MED_RULES_PATH"], "|", CONFIG["ALLERGIES_PATH"])


Meds/allergy paths → /mnt/data/interaction_rules.json | /mnt/data/patient_allergies.json


In [89]:
# === Calc utils (shared, model-free) ===
from typing import Dict, Any, Optional, List, Tuple
import re, math
from datetime import datetime

def _pick(d: Dict[str, Any], names: List[str], cast=float, default=None):
    for n in names:
        if n in d and d[n] not in (None, ""):
            try:
                return cast(d[n])
            except Exception:
                try:
                    return cast(str(d[n]).replace(",", "."))
                except Exception:
                    pass
    return default

def _bool(v) -> bool:
    if isinstance(v, bool): return v
    s = str(v).strip().lower()
    return s in {"1","true","yes","y","ja","on","oui"}

def _safe_round(x, nd=1):
    try:
        return round(float(x), nd)
    except Exception:
        return x
# === Phase-2 calculators A (core scores; model-free) ===
from typing import Dict, Any

def calc_qsofa(v: Dict[str,Any]):
    rr=_pick(v,["rr","resp_rate"]); sbp=_pick(v,["sbp","systolic"]); gcs=_pick(v,["gcs"],float)
    avpu = (v.get("avpu") or "").upper()[:1]
    altered = (gcs is not None and gcs<15) or avpu in {"V","P","U"}
    return {"name":"qSOFA","score": int((rr is not None and rr>=22)) + int((sbp is not None and sbp<=100)) + int(altered)}

def calc_mews(v: Dict[str,Any]):
    def rr_s(x):  return 3 if x is not None and x<=8 else (0 if x and 9<=x<=14 else (1 if x and 15<=x<=20 else (2 if x and 21<=x<=29 else (3 if x and x>=30 else 0))))
    def hr_s(x):  return 2 if x is not None and x<=40 else (1 if x and 41<=x<=50 else (0 if x and 51<=x<=100 else (1 if x and 101<=x<=110 else (2 if x and 111<=x<=129 else (3 if x and x>=130 else 0)))))
    def sbp_s(x): return 3 if x is not None and x<=70 else (2 if x and 71<=x<=80 else (1 if x and 81<=x<=100 else (0 if x and 101<=x<=199 else (2 if x and x>=200 else 0))))
    def t_s(x):   return 2 if x is not None and x<=35.0 else (1 if x and 35.1<=x<=36.0 else (0 if x and 36.1<=x<=38.0 else (1 if x and 38.1<=x<=38.5 else (2 if x and x>=38.6 else 0))))
    def avpu_s(x): return {"A":0,"V":1,"P":2,"U":3}.get((x or "A").upper()[:1],0)
    return {"name":"MEWS","score": int(rr_s(_pick(v,["rr"]))+hr_s(_pick(v,["hr","pulse"]))+sbp_s(_pick(v,["sbp"]))+t_s(_pick(v,["temp"]))+avpu_s(v.get("avpu")))}

def calc_heart(p: Dict[str,Any]):
    age=_pick(p,["age"],int); hist=_pick(p,["heart_history"],int); ecg=_pick(p,["heart_ecg"],int); risk=_pick(p,["heart_risk"],int)
    trop=_pick(p,["troponin","hs_troponin","trop"]); uln=_pick(p,["troponin_uln"],float)
    age_s = 2 if (age is not None and age>=65) else (1 if (age is not None and 45<=age<=64) else 0)
    ratio = (trop/uln) if (trop is not None and uln) else None
    trop_s = 2 if (ratio is not None and ratio>3) else (1 if (ratio is not None and 1<ratio<=3) else 0)
    total = (hist or 0)+(ecg or 0)+age_s+(risk or 0)+trop_s
    return {"name":"HEART","score": int(total)}

def calc_grace_coarse(p: Dict[str,Any]):
    age=_pick(p,["age"],int); hr=_pick(p,["hr","pulse"]); sbp=_pick(p,["sbp"]); crea=_pick(p,["creatinine"])
    sc=0
    if age is not None: sc += (0 if age<40 else 20 if age<60 else 40 if age<80 else 60)
    if hr  is not None: sc += (0 if hr<70  else 10 if hr<90  else 20 if hr<110 else 30 if hr<150 else 40)
    if sbp is not None: sc += (40 if sbp<80 else 30 if sbp<100 else 10 if sbp<120 else 0)
    if crea is not None: sc += (0 if crea<1.2 else 10 if crea<2.0 else 20 if crea<3.0 else 30)
    return {"name":"GRACE_coarse","score": int(sc)}

def calc_sofa_min(v: Dict[str,Any], labs: Dict[str,Any]):
    pf=None; pao2=_pick(labs,["pao2"]); fio2=_pick(labs,["fio2"])
    if pao2 is not None and fio2:
        try: pf=float(pao2)/float(fio2)
        except Exception: pf=None
    plate=_pick(labs,["platelets","plt"]); bili=_pick(labs,["bilirubin"]); mapv=_pick(v,["map"]); gcs=_pick(v,["gcs"]); crea=_pick(labs,["creatinine"])
    sc=0
    if pf is not None:    sc += (4 if pf<100 else 3 if pf<200 else 2 if pf<300 else 1 if pf<400 else 0)
    if plate is not None: sc += (4 if plate<20 else 3 if plate<50 else 2 if plate<100 else 1 if plate<150 else 0)
    if bili  is not None: sc += (4 if bili>=12 else 3 if bili>=6 else 2 if bili>=2 else 1 if bili>=1.2 else 0)
    if mapv  is not None: sc += (1 if mapv<70 else 0)  # presence-guarded (MAP not required)
    if gcs   is not None: sc += (4 if gcs<6 else 3 if gcs<10 else 2 if gcs<13 else 1 if gcs<15 else 0)
    if crea  is not None: sc += (4 if crea>=5 else 3 if crea>=3.5 else 2 if crea>=2 else 1 if crea>=1.2 else 0)
    return {"name":"SOFA_min","score": int(sc)}

def calc_sirs(p: Dict[str,Any]):
    crit = {
        "temp>38/<36": int(((_pick(p,["temp"],float) or 37)>38) or ((_pick(p,["temp"],float) or 37)<36)),
        "hr>90": int((_pick(p,["hr","pulse"],float) or 0) > 90),
        "rr>20/paco2<32": int(((_pick(p,["rr"],float) or 0) > 20) or ((_pick(p,["paco2"],float) or 100) < 32)),
        "wbc>12/<4/bands>10%": int(((_pick(p,["wbc"],float) or 7) > 12) or ((_pick(p,["wbc"],float) or 7) < 4) or ((_pick(p,["bands_pct"],float) or 0) > 10)),
    }
    return {"name":"SIRS","score": int(sum(crit.values()))}

def sepsis3_screen(v: Dict[str,Any], labs: Dict[str,Any], ctx: Dict[str,Any], sofa_min: Dict[str,Any], qsofa: Dict[str,Any]):
    sus = _bool(ctx.get("suspected_infection")); on_pressors=_bool(ctx.get("vasopressors"))
    mapv=_pick(v,["map"]); lact=_pick(labs,["lactate"])
    septic_shock = bool((mapv is not None and mapv<65) and (lact is not None and lact>2) and on_pressors)
    return {"name":"SEPSIS3","sepsis_flag": bool(sus and sofa_min["score"]>=2), "septic_shock": septic_shock}


In [90]:
# === Phase-2 calculators A (core scores; model-free) ===
from typing import Dict, Any

def calc_qsofa(v: Dict[str,Any]):
    rr=_pick(v,["rr","resp_rate"]); sbp=_pick(v,["sbp","systolic"]); gcs=_pick(v,["gcs"],float)
    avpu = (v.get("avpu") or "").upper()[:1]
    altered = (gcs is not None and gcs<15) or avpu in {"V","P","U"}
    return {"name":"qSOFA","score": int((rr is not None and rr>=22)) + int((sbp is not None and sbp<=100)) + int(altered)}

def calc_mews(v: Dict[str,Any]):
    def rr_s(x):  return 3 if x is not None and x<=8 else (0 if x and 9<=x<=14 else (1 if x and 15<=x<=20 else (2 if x and 21<=x<=29 else (3 if x and x>=30 else 0))))
    def hr_s(x):  return 2 if x is not None and x<=40 else (1 if x and 41<=x<=50 else (0 if x and 51<=x<=100 else (1 if x and 101<=x<=110 else (2 if x and 111<=x<=129 else (3 if x and x>=130 else 0)))))
    def sbp_s(x): return 3 if x is not None and x<=70 else (2 if x and 71<=x<=80 else (1 if x and 81<=x<=100 else (0 if x and 101<=x<=199 else (2 if x and x>=200 else 0))))
    def t_s(x):   return 2 if x is not None and x<=35.0 else (1 if x and 35.1<=x<=36.0 else (0 if x and 36.1<=x<=38.0 else (1 if x and 38.1<=x<=38.5 else (2 if x and x>=38.6 else 0))))
    def avpu_s(x): return {"A":0,"V":1,"P":2,"U":3}.get((x or "A").upper()[:1],0)
    return {"name":"MEWS","score": int(rr_s(_pick(v,["rr"]))+hr_s(_pick(v,["hr","pulse"]))+sbp_s(_pick(v,["sbp"]))+t_s(_pick(v,["temp"]))+avpu_s(v.get("avpu")))}

def calc_heart(p: Dict[str,Any]):
    age=_pick(p,["age"],int); hist=_pick(p,["heart_history"],int); ecg=_pick(p,["heart_ecg"],int); risk=_pick(p,["heart_risk"],int)
    trop=_pick(p,["troponin","hs_troponin","trop"]); uln=_pick(p,["troponin_uln"],float)
    age_s = 2 if (age is not None and age>=65) else (1 if (age is not None and 45<=age<=64) else 0)
    ratio = (trop/uln) if (trop is not None and uln) else None
    trop_s = 2 if (ratio is not None and ratio>3) else (1 if (ratio is not None and 1<ratio<=3) else 0)
    total = (hist or 0)+(ecg or 0)+age_s+(risk or 0)+trop_s
    return {"name":"HEART","score": int(total)}

def calc_grace_coarse(p: Dict[str,Any]):
    age=_pick(p,["age"],int); hr=_pick(p,["hr","pulse"]); sbp=_pick(p,["sbp"]); crea=_pick(p,["creatinine"])
    sc=0
    if age is not None: sc += (0 if age<40 else 20 if age<60 else 40 if age<80 else 60)
    if hr  is not None: sc += (0 if hr<70  else 10 if hr<90  else 20 if hr<110 else 30 if hr<150 else 40)
    if sbp is not None: sc += (40 if sbp<80 else 30 if sbp<100 else 10 if sbp<120 else 0)
    if crea is not None: sc += (0 if crea<1.2 else 10 if crea<2.0 else 20 if crea<3.0 else 30)
    return {"name":"GRACE_coarse","score": int(sc)}

def calc_sofa_min(v: Dict[str,Any], labs: Dict[str,Any]):
    pf=None; pao2=_pick(labs,["pao2"]); fio2=_pick(labs,["fio2"])
    if pao2 is not None and fio2:
        try: pf=float(pao2)/float(fio2)
        except Exception: pf=None
    plate=_pick(labs,["platelets","plt"]); bili=_pick(labs,["bilirubin"]); mapv=_pick(v,["map"]); gcs=_pick(v,["gcs"]); crea=_pick(labs,["creatinine"])
    sc=0
    if pf is not None:    sc += (4 if pf<100 else 3 if pf<200 else 2 if pf<300 else 1 if pf<400 else 0)
    if plate is not None: sc += (4 if plate<20 else 3 if plate<50 else 2 if plate<100 else 1 if plate<150 else 0)
    if bili  is not None: sc += (4 if bili>=12 else 3 if bili>=6 else 2 if bili>=2 else 1 if bili>=1.2 else 0)
    if mapv  is not None: sc += (1 if mapv<70 else 0)  # presence-guarded (MAP not required)
    if gcs   is not None: sc += (4 if gcs<6 else 3 if gcs<10 else 2 if gcs<13 else 1 if gcs<15 else 0)
    if crea  is not None: sc += (4 if crea>=5 else 3 if crea>=3.5 else 2 if crea>=2 else 1 if crea>=1.2 else 0)
    return {"name":"SOFA_min","score": int(sc)}

def calc_sirs(p: Dict[str,Any]):
    crit = {
        "temp>38/<36": int(((_pick(p,["temp"],float) or 37)>38) or ((_pick(p,["temp"],float) or 37)<36)),
        "hr>90": int((_pick(p,["hr","pulse"],float) or 0) > 90),
        "rr>20/paco2<32": int(((_pick(p,["rr"],float) or 0) > 20) or ((_pick(p,["paco2"],float) or 100) < 32)),
        "wbc>12/<4/bands>10%": int(((_pick(p,["wbc"],float) or 7) > 12) or ((_pick(p,["wbc"],float) or 7) < 4) or ((_pick(p,["bands_pct"],float) or 0) > 10)),
    }
    return {"name":"SIRS","score": int(sum(crit.values()))}

def sepsis3_screen(v: Dict[str,Any], labs: Dict[str,Any], ctx: Dict[str,Any], sofa_min: Dict[str,Any], qsofa: Dict[str,Any]):
    sus = _bool(ctx.get("suspected_infection")); on_pressors=_bool(ctx.get("vasopressors"))
    mapv=_pick(v,["map"]); lact=_pick(labs,["lactate"])
    septic_shock = bool((mapv is not None and mapv<65) and (lact is not None and lact>2) and on_pressors)
    return {"name":"SEPSIS3","sepsis_flag": bool(sus and sofa_min["score"]>=2), "septic_shock": septic_shock}


In [91]:
# === Phase-2 calculators B (PE/VTE, GI/hepatic, corrections + lab assess) ===
from typing import Dict, Any, List, Optional, Tuple

def calc_wells_pe(p: Dict[str,Any]):
    pts=0.0
    pts+=3.0 if _bool(p.get("dvt_signs")) else 0.0
    pts+=3.0 if _bool(p.get("pe_most_likely")) else 0.0
    pts+=1.5 if ((_pick(p,["hr","pulse"],float) or 0)>100) else 0.0
    pts+=1.5 if (_bool(p.get("immobilized")) or _bool(p.get("recent_surgery_4w"))) else 0.0
    pts+=1.5 if _bool(p.get("prev_vte")) else 0.0
    pts+=1.0 if _bool(p.get("hemoptysis")) else 0.0
    pts+=1.0 if _bool(p.get("cancer_active")) else 0.0
    return {"name":"WELLS_PE","score": pts, "tier2": ("likely" if pts>4 else "unlikely")}

def calc_wells_dvt(p: Dict[str,Any]):
    pts=0
    pts+=1 if _bool(p.get("cancer_active")) else 0
    pts+=1 if (_bool(p.get("paresis")) or _bool(p.get("plaster_cast"))) else 0
    pts+=1 if (_bool(p.get("bedridden_3d")) or _bool(p.get("surgery_12w"))) else 0
    pts+=1 if _bool(p.get("deep_vein_tenderness")) else 0
    pts+=1 if _bool(p.get("entire_leg_swollen")) else 0
    pts+=1 if _bool(p.get("calf_swelling_gt3cm")) else 0
    pts+=1 if _bool(p.get("pitting_edema")) else 0
    pts+=1 if _bool(p.get("collateral_nonvaricose")) else 0
    pts+=1 if _bool(p.get("prev_dvt")) else 0
    pts-=2 if _bool(p.get("alt_dx_as_likely")) else 0
    return {"name":"WELLS_DVT","score": int(pts), "tier2": ("likely" if pts>=2 else "unlikely")}

def calc_perc(p: Dict[str,Any]):
    crit = {
        "age<50": int((_pick(p,["age"],int) or 10) < 50),
        "hr<100": int((_pick(p,["hr","pulse"],float) or 0) < 100),
        "sao2>=95": int((_pick(p,["sao2","spo2"],float) or 0) >= 95),
        "no_hemoptysis": int(not _bool(p.get("hemoptysis"))),
        "no_estrogen": int(not _bool(p.get("estrogen_use"))),
        "no_surg/trauma_4w": int(not (_bool(p.get("recent_surgery_4w")) or _bool(p.get("recent_trauma_4w")))),
        "no_prior_vte": int(not _bool(p.get("prev_vte"))),
        "no_unilateral_swelling": int(not _bool(p.get("unilateral_leg_swelling"))),
    }
    return {"name":"PERC","passed": bool(all(crit.values()))}

def calc_pesi(p: Dict[str,Any]):
    age=_pick(p,["age"],int) or 0
    male = 10 if (str(p.get("sex") or "").upper().startswith("M")) else 0
    cancer = 30 if _bool(p.get("cancer_active")) else 0
    hf = 10 if _bool(p.get("heart_failure")) else 0
    lung = 10 if (_bool(p.get("copd")) or _bool(p.get("chronic_lung_disease"))) else 0
    hr = 20 if ((_pick(p,["hr","pulse"],float) or 0) >=110) else 0
    sbp = 30 if ((_pick(p,["sbp"],float) or 200) < 100) else 0
    rr = 20 if ((_pick(p,["rr"],float) or 0) >=30) else 0
    temp = 20 if ((_pick(p,["temp"],float) or 37) < 36) else 0
    altered = 60 if ((_pick(p,["gcs"],float) or 15) < 15) else 0
    sat = 20 if ((_pick(p,["sao2","spo2"],float) or 100) < 90) else 0
    score = age+male+cancer+hf+lung+hr+sbp+rr+temp+altered+sat
    klass = "I" if score<=65 else "II" if score<=85 else "III" if score<=105 else "IV" if score<=125 else "V"
    return {"name":"PESI","score": int(score), "class": klass}

def calc_spesi(p: Dict[str,Any]):
    comps = {
        "age>80": int((_pick(p,["age"],int) or 0) > 80),
        "cancer": int(_bool(p.get("cancer_active"))),
        "cardiopulm": int(_bool(p.get("heart_failure")) or _bool(p.get("copd")) or _bool(p.get("chronic_lung_disease"))),
        "hr>=110": int((_pick(p,["hr","pulse"],float) or 0) >= 110),
        "sbp<100": int((_pick(p,["sbp"],float) or 200) < 100),
        "o2<90": int((_pick(p,["sao2","spo2"],float) or 100) < 90),
    }
    return {"name":"sPESI","score": int(sum(comps.values()))}

def calc_marburg(p: Dict[str,Any]):
    sex=(str(p.get("sex") or "")[:1]).upper(); age=_pick(p,["age"],int)
    vasc=_bool(p.get("vasc_disease")); exert=_bool(p.get("exertional")); pt_thinks=_bool(p.get("patient_assumes_cardiac"))
    palp = p.get("palpation_reproducible"); not_repro = (palp is False)
    age_sex = ((sex=="M" and age is not None and age>=55) or (sex=="F" and age is not None and age>=65))
    sc = int(bool(age_sex)) + int(vasc) + int(exert) + int(pt_thinks) + int(bool(not_repro))
    return {"name":"MARBURG","score": int(sc)}

def calc_gbs(p: Dict[str,Any]):
    score=0
    urea=_pick(p,["urea_mmol_l","urea"]); bun=_pick(p,["bun_mg_dl"])
    if urea is None and bun is not None: urea=float(bun)/2.8
    hb_gL=_pick(p,["hb_g_l"]); 
    if hb_gL is None:
        hb_gdl=_pick(p,["hb","hb_g_dl"]); 
        if hb_gdl is not None: hb_gL=hb_gdl*10.0
    sbp=_pick(p,["sbp"]); hr=_pick(p,["hr","pulse"]); male = str(p.get("sex") or "").upper().startswith("M")
    if urea is not None:
        score += 2 if 6.5<=urea<=7.9 else 0; score += 3 if 8.0<=urea<=9.9 else 0
        score += 4 if 10.0<=urea<=25.0 else 0; score += 6 if urea>25.0 else 0
    if hb_gL is not None:
        if male:   score += (1 if 120<=hb_gL<=129 else 0) + (3 if 100<=hb_gL<=119 else 0) + (6 if hb_gL<100 else 0)
        else:      score += (1 if 100<=hb_gL<=119 else 0) + (6 if hb_gL<100 else 0)
    if sbp is not None: score += (1 if 100<=sbp<=109 else 0) + (2 if 90<=sbp<=99 else 0) + (3 if sbp<90 else 0)
    score += 1 if (hr is not None and hr>=100) else 0
    score += 1 if _bool(p.get("melena")) else 0
    score += 2 if _bool(p.get("syncope")) else 0
    score += 2 if _bool(p.get("hepatic_disease")) else 0
    score += 2 if _bool(p.get("cardiac_failure")) else 0
    return {"name":"GBS","score": int(score)}

def calc_child_pugh(p: Dict[str,Any]):
    bili=_pick(p,["bilirubin"]); alb=_pick(p,["albumin"]); inr=_pick(p,["inr"])
    asc=(p.get("ascites") or "").lower(); ence=(p.get("encephalopathy") or "").lower()
    sc=0; filled=0
    if bili is not None: sc+=(1 if bili<2 else 2 if bili<=3 else 3); filled+=1
    if alb  is not None: sc+=(1 if alb>3.5 else 2 if alb>=2.8 else 3); filled+=1
    if inr  is not None: sc+=(1 if inr<1.7 else 2 if inr<=2.3 else 3); filled+=1
    if asc:              sc+=(1 if asc.startswith("n") else 2 if asc.startswith(("mild","slight")) else 3); filled+=1
    if ence:             sc+=(1 if ence in {"none","0"} else 2 if any(x in ence for x in ["1","2","i","ii"]) else 3); filled+=1
    if filled<5: return {"name":"CHILD_PUGH","score_partial": int(sc), "class":"incomplete"}
    klass = "A" if sc<=6 else ("B" if sc<=9 else "C")
    return {"name":"CHILD_PUGH","score": int(sc), "class": klass}

def corrected_calcium(total_ca, albumin, units="mg/dL", normal_alb=None):
    if total_ca is None or albumin is None: return None
    if "mmol" in (units or "").lower():
        alb_gl = albumin if albumin>10 else albumin*10.0
        normal = 40.0 if normal_alb is None else float(normal_alb)
        return float(total_ca) + 0.02*(normal - alb_gl)
    normal = 4.0 if normal_alb is None else float(normal_alb)
    return float(total_ca) + 0.8*(normal - float(albumin))

def anion_gap(na, cl, hco3, k=None, albumin_gdl=None):
    if na is None or cl is None or hco3 is None: return None
    ag = (float(na)+(float(k) if k is not None else 0.0)) - (float(cl)+float(hco3))
    agc = ag + (2.5*(4.0 - float(albumin_gdl))) if albumin_gdl is not None else None
    return {"ag": _safe_round(ag,1), "ag_albumin_corrected": _safe_round(agc,1) if agc is not None else None}

# --- Lab assessors (reuse HL7 unit converters if available; else safe fallback) ---

def assess_d_dimer(value, unit, age, pregnant=False):
    if value is None: return {"available": False}
    # Prefer canonical converter from HL7 cell if present
    conv = globals().get("_to_ddimer_mgL_feu", None)
    if conv:
        mgL = conv(value, unit); val_ug = mgL*1000.0 if mgL is not None else None
    else:
        unit_l = (unit or "").lower()
        # fallback assumes FEU; micro→mg
        if "mg/l" in unit_l: val_ug = float(value)*1000.0
        else:                val_ug = float(value)  # assume μg/L FEU
    thr = 500.0
    if age is not None and age>50 and not pregnant:
        thr = float(age)*10.0
    return {"available": True, "value_ug_per_l": val_ug, "thr_ug_per_l": thr, "ok_below_thr": bool(val_ug < thr)}

def assess_troponin_delta(series: List[Tuple[Optional[datetime], float, str]]):
    if not series or len(series)<2: return {"available": False}
    try:
        series = sorted(series, key=lambda x: (x[0] or datetime.min))
    except Exception:
        pass
    (t0,v0,u0),(t1,v1,u1)=series[-2],series[-1]
    # Prefer canonical converter from HL7 cell if present
    conv = globals().get("_to_trop_ngL", None)
    def _fallback_to_ngL(v,u):
        u=(u or "").lower()
        if "ng/ml" in u: return float(v)*1000.0
        if "µg/l" in u or "ug/l" in u: return float(v)*1000.0
        return float(v)  # assume ng/L
    to_ngL = conv or _fallback_to_ngL
    p=to_ngL(v0,u0); c=to_ngL(v1,u1); d=c-p; pct=(abs(d)/p*100.0) if p not in (None,0) else None
    flag = (p is not None and ((abs(d)>=51.0) or (pct is not None and pct>=20.0)))
    return {"available": True, "prev": p, "curr": c, "delta_abs": d, "delta_pct": pct, "flag": bool(flag)}


In [92]:
# === Phase-2 bundle helpers (lab normalization + context) ===
from typing import Dict, Any, Optional, List, Tuple

def build_labs_norm(labs: Dict[str, Any]) -> Dict[str, Any]:
    """
    Map canonical HL7 keys to calculator-friendly names (presence-guarded).
    Does NOT mutate input; returns a new dict.
    """
    labs = dict(labs or {})
    out = dict(labs)

    # Bilirubin: µmol/L (canonical) → mg/dL (calculators)
    try:
        if "lab_bili_total_umolL" in labs and "bilirubin" not in out:
            out["bilirubin"] = float(labs["lab_bili_total_umolL"]) / 17.104
    except Exception:
        pass

    # Albumin: g/L (canonical) → g/dL (calculators)
    try:
        if "lab_albumin_gL" in labs:
            alb_gdl = float(labs["lab_albumin_gL"]) / 10.0
            out.setdefault("albumin", alb_gdl)
            out.setdefault("albumin_g_dl", alb_gdl)
    except Exception:
        pass

    # Electrolytes for anion gap; lactate for Sepsis3 (prefer explicit; fall back to VBG)
    if "na" not in out and "vbg_Na" in labs: out["na"] = labs["vbg_Na"]
    if "cl" not in out and "vbg_Cl" in labs: out["cl"] = labs["vbg_Cl"]
    if "hco3" not in out and "vbg_HCO3" in labs: out["hco3"] = labs["vbg_HCO3"]
    if "lactate" not in out and "vbg_lactate" in labs: out["lactate"] = labs["vbg_lactate"]

    return out

def _ctx_age_pregnant(vitals: Dict[str,Any], context: Optional[Dict[str,Any]]) -> tuple[Optional[int], bool]:
    context = context or {}
    age = _pick({**vitals, **context}, ["age"], int)
    pregnant = bool(context.get("pregnant", False))
    return age, pregnant


In [93]:
# === Vitals normalizer (model-free, presence-guarded) ===
import re, math
import pandas as pd
from typing import Dict, Any, Optional

_VITAL_SYNONYMS = {
    "SBP":      ["SBP","RR_sys","BP_sys","systolic","Systolic"],
    "DBP":      ["DBP","RR_dia","BP_dia","diastolic","Diastolic"],
    "HR":       ["HR","HF","heart_rate"],
    "SpO2":     ["SpO2","SpO₂","SpO2_pct","Sats"],
    "RR_rate":  ["RR_rate","AF","Resp","respiratory_rate"],  # breaths/min (not blood pressure!)
    "Temp_C":   ["Temp_C","Temp","temperature_c","T"],
    "GCS":      ["GCS"],
    "RR_str":   ["RR","RR_mmHg","RR_str"],                    # e.g. "120/80"
}

def _first(fd: Dict[str,Any], keys) -> Optional[Any]:
    for k in keys:
        if k in fd and fd[k] is not None:
            return fd[k]
    return None

def _parse_bp_pair(text: str) -> Optional[tuple]:
    m = re.search(r"(\d{2,3})\s*/\s*(\d{2,3})", str(text))
    if m: return float(m.group(1)), float(m.group(2))
    return None

def normalize_vitals(raw: Dict[str,Any], now_utc: Optional[pd.Timestamp]=None) -> Dict[str,Any]:
    """
    Returns canonical vitals independent of any model bundle:
    SBP, DBP, HR, SpO2, RR_rate, Temp_C, GCS, shock_index, hypotension_flag, hypoxia_flag, since_vitals_min.
    """
    out: Dict[str,Any] = {}
    # SBP/DBP from direct fields or "RR '120/80'"
    sbp = _first(raw, _VITAL_SYNONYMS["SBP"])
    dbp = _first(raw, _VITAL_SYNONYMS["DBP"])
    rr_pair = _first(raw, _VITAL_SYNONYMS["RR_str"])
    if (sbp is None or dbp is None) and rr_pair is not None:
        p = _parse_bp_pair(rr_pair)
        if p: sbp = sbp or p[0]; dbp = dbp or p[1]
    out["SBP"]  = float(sbp) if sbp is not None else None
    out["DBP"]  = float(dbp) if dbp is not None else None

    # Other primary vitals
    for canon, keys in [("HR","HR"),("SpO2","SpO2"),("RR_rate","RR_rate"),("Temp_C","Temp_C"),("GCS","GCS")]:
        val = _first(raw, _VITAL_SYNONYMS[canon])
        out[canon] = float(val) if val is not None else None

    # Derived, model-free
    hr  = out.get("HR"); sbp = out.get("SBP"); spo2 = out.get("SpO2")
    out["shock_index"]     = (hr/sbp) if (isinstance(hr,(int,float)) and isinstance(sbp,(int,float)) and sbp>0) else None
    out["hypotension_flag"]= (sbp is not None and sbp < 90)
    out["hypoxia_flag"]    = (spo2 is not None and spo2 < 90)

    # Age/sex NEVER used here (bias hygiene); gates don’t touch protected attrs.
    # since_vitals_min: prefer provided timestamp, else compute from 'vitals_ts' or now
    now = now_utc or pd.Timestamp.utcnow()
    ts  = raw.get("vitals_ts") or raw.get("ts")  # accept either if present
    try:
        tsv = pd.to_datetime(ts) if ts is not None else None
    except Exception:
        tsv = None
    out["since_vitals_min"] = (now - tsv).total_seconds()/60.0 if tsv is not None else None
    return out


In [94]:
# === Phase-2 bundle core scores (pure) ===
from typing import Dict, Any, Optional

def compute_phase2_scores(vitals: Dict[str,Any], labs_norm: Dict[str,Any], context: Optional[Dict[str,Any]]=None) -> Dict[str,Any]:
    context = context or {}

    s_qsofa = calc_qsoFA(vitals) if 'calc_qsoFA' in globals() else calc_qsofa(vitals)  # tolerate minor naming diffs
    s_mews  = calc_mews(vitals)
    s_heart = calc_heart({**vitals, **labs_norm})
    s_grace = calc_grace_coarse({**vitals, **labs_norm})
    s_sofa  = calc_sofa_min(vitals, labs_norm)
    sirs    = calc_sirs({**vitals, **labs_norm})
    sepsis3 = sepsis3_screen(vitals, labs_norm, context, s_sofa, s_qsofa)

    wells_pe  = calc_wells_pe({**vitals, **labs_norm, **context})
    wells_dvt = calc_wells_dvt({**vitals, **labs_norm, **context})
    perc      = calc_perc({**vitals, **labs_norm, **context})
    pesi      = calc_pesi({**vitals, **labs_norm, **context})
    spesi     = calc_spesi({**vitals, **labs_norm, **context})
    marburg   = calc_marburg({**vitals, **labs_norm, **context})
    gbs       = calc_gbs({**vitals, **labs_norm, **context})
    childpugh = calc_child_pugh({**vitals, **labs_norm, **context})

    return {
        "qsofa": s_qsofa, "mews": s_mews, "heart": s_heart, "grace": s_grace, "sofa": s_sofa,
        "sirs": sirs, "sepsis3": sepsis3,
        "wells_pe": wells_pe, "wells_dvt": wells_dvt, "perc": perc,
        "pesi": pesi, "spesi": spesi,
        "marburg": marburg, "gbs": gbs, "childpugh": childpugh
    }


In [95]:
# === Phase-2 bundle (assembler) ===
from typing import Dict, Any, Optional, List, Tuple
from datetime import datetime

def phase2_bundle(vitals: Dict[str,Any], labs: Dict[str,Any], context: Optional[Dict[str,Any]]=None) -> Dict[str,Any]:
    context = context or {}
    labs_norm = build_labs_norm(labs)
    age, pregnant = _ctx_age_pregnant(vitals, context)

    # Scores
    sc = compute_phase2_scores(vitals, labs_norm, context)
    s_qsofa=sc["qsofa"]; s_mews=sc["mews"]; s_heart=sc["heart"]; s_grace=sc["grace"]; s_sofa=sc["sofa"]
    sirs=sc["sirs"]; sepsis3=sc["sepsis3"]
    wells_pe=sc["wells_pe"]; wells_dvt=sc["wells_dvt"]; perc=sc["perc"]
    pesi=sc["pesi"]; spesi=sc["spesi"]; marburg=sc["marburg"]; gbs=sc["gbs"]; childpugh=sc["childpugh"]

    # D-dimer (canonical mg/L FEU or legacy µg/L FEU)
    if "lab_ddimer_mgL_FEU" in labs_norm:
        d_val = float(labs_norm["lab_ddimer_mgL_FEU"]); d_unit = "mg/L FEU"
    else:
        d_val  = _pick(labs_norm, ["d_dimer_feu_ug_l","d_dimer","ddimer","d-dimer"])
        d_unit = labs_norm.get("d_dimer_unit") or "µg/L FEU"
    d_assess = assess_d_dimer(d_val, d_unit, age, pregnant) if d_val is not None else {"available": False}

    # Troponin Δ (legacy series only)
    series: List[Tuple[Optional[datetime], float, str]] = []
    for it in (labs_norm.get("troponin_series") or []):
        if isinstance(it, dict):
            ts=None
            if it.get("time"):
                try: ts = datetime.fromisoformat(str(it["time"]).replace("Z",""))
                except Exception: ts=None
            series.append((ts, _pick(it, ["value","val"]), it.get("unit","ng/L")))
    t_assess = assess_troponin_delta(series) if len(series) >= 2 else {"available": False}

    # Corrections
    ca_corr = corrected_calcium(
        _pick(labs_norm, ["calcium","ca","calcium_mg_dl"]),
        _pick(labs_norm, ["albumin","alb","albumin_g_dl"]),
        units="mg/dL"
    )
    ag_val  = anion_gap(
        _pick(labs_norm, ["na","sodium"]),
        _pick(labs_norm, ["cl","chloride"]),
        _pick(labs_norm, ["hco3","bicarbonate"]),
        k=_pick(labs_norm, ["k","potassium"]),
        albumin_gdl=_pick(labs_norm, ["albumin","alb","albumin_g_dl"])
    )

    # One-liners
    ones: List[str] = []
    ones.append(f"qSOFA={s_qsofa['score']}  MEWS={s_mews['score']}  GRACE(coarse)={s_grace['score']}")
    ones.append(f"HEART={s_heart['score']}  SOFA(min)={s_sofa['score']}")
    if d_assess.get("available"):
        ones.append(f"D-dimer {int(d_assess['value_ug_per_l'])} vs thr {int(d_assess['thr_ug_per_l'])} μg/L → "
                    f"{'OK' if d_assess['ok_below_thr'] else 'High'}")
    if t_assess.get("available"):
        da=_safe_round(t_assess['delta_abs'],1)
        dp=_safe_round(t_assess['delta_pct'],1) if t_assess['delta_pct'] is not None else None
        ones.append(f"Troponin Δ {da} ng/L ({dp}%) → {'FLAG' if t_assess['flag'] else 'ok'}")
    ones.append(f"Wells-PE={_safe_round(wells_pe['score'],1)} ({wells_pe['tier2']})  "
                f"Wells-DVT={wells_dvt['score']} ({wells_dvt['tier2']})")
    ones.append(f"PERC={'pass' if perc['passed'] else 'fail'}  sPESI={spesi['score']}  PESI={pesi['class']}/{pesi['score']}")
    ones.append(f"SIRS={sirs['score']}  Sepsis3: {'YES' if sepsis3['sepsis_flag'] else 'no'}  "
                f"Shock: {'YES' if sepsis3['septic_shock'] else 'no'}")
    ones.append(f"MARBURG={marburg['score']}/5  GBS={gbs['score']}")
    if childpugh.get("class") == "incomplete":
        ones.append("Child-Pugh incomplete (need 5/5 inputs)")
    else:
        ones.append(f"Child-Pugh {childpugh['class']} ({childpugh['score']})")
    if ca_corr is not None:
        ones.append(f"Corrected Ca={_safe_round(ca_corr,2)} mg/dL")
    if ag_val is not None:
        ab = f", AGcorr={ag_val['ag_albumin_corrected']}" if ag_val.get("ag_albumin_corrected") is not None else ""
        ones.append(f"Anion gap={ag_val['ag']}{ab}")

    return {
        "one_liners": ones,
        "scores": {
            "qsofa": s_qsofa["score"], "mews": s_mews["score"], "heart": s_heart["score"], "grace_coarse": s_grace["score"],
            "sofa_min": s_sofa["score"], "wells_pe": wells_pe["score"], "wells_dvt": wells_dvt["score"],
            "pesi": pesi["score"], "spesi": spesi["score"], "sirs": sirs["score"],
            "sepsis3": int(sepsis3["sepsis_flag"]), "gbs": gbs["score"], "marburg": marburg["score"],
            "child_pugh": childpugh.get("score") or childpugh.get("score_partial")
        },
        "labs": {
            "d_dimer": (d_assess.get("value_ug_per_l") if d_assess.get("available") else None),
            "d_dimer_thr": d_assess.get("thr_ug_per_l"),
            "trop_prev": t_assess.get("prev"), "trop_curr": t_assess.get("curr"),
            "trop_delta": t_assess.get("delta_abs"), "trop_delta_pct": t_assess.get("delta_pct"),
            "trop_flag": t_assess.get("flag")
        }
    }


In [96]:
# ---- UI (ipywidgets) + HL7 merge ----
if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W, pandas as pd, json
        EVENT_LOG_PATH = CONFIG.get("EVENT_LOG_PATH", "/mnt/data/event_log.jsonl")

        # --- Text areas ---
        vitals_in = W.Textarea(
            value='{"rr":24,"sbp":95,"gcs":14,"hr":120,"temp":38.6,"avpu":"V","map":65,"age":68,"sex":"M","sao2":97}',
            description="Vitals JSON", layout=W.Layout(width="100%", height="90px")
        )
        labs_in = W.Textarea(
            value='{"d_dimer":780,"d_dimer_unit":"μg/L FEU","troponin_series":[{"time":"2025-08-20T05:00:00Z","value":18,"unit":"ng/L"},{"time":"2025-08-20T09:00:00Z","value":78,"unit":"ng/L"}],"creatinine":1.8,"platelets":95,"albumin":2.8,"calcium":7.9,"na":138,"cl":105,"hco3":20,"bilirubin":2.1,"inr":1.9}',
            description="Labs JSON", layout=W.Layout(width="100%", height="130px")
        )
        ctx_in = W.Textarea(
            value='{"pregnant": false, "suspected_infection": true, "vasopressors": false, "pe_most_likely": true, "dvt_signs": false, "ascites":"mild","encephalopathy":"1-2"}',
            description="Context JSON", layout=W.Layout(width="100%", height="90px")
        )

        # --- HEART component inputs (0–2) ---
        heart_hist = W.Dropdown(options=[0,1,2], value=0, description="HEART: History")
        heart_ecg  = W.Dropdown(options=[0,1,2], value=0, description="HEART: ECG")
        heart_risk = W.Dropdown(options=[0,1,2], value=0, description="HEART: Risk")

        # --- PE / PERC flags ---
        chk_pe_most  = W.Checkbox(description="PE most likely", value=True)
        chk_dvt_signs= W.Checkbox(description="DVT signs", value=False)
        chk_immob    = W.Checkbox(description="Immobilized / recent surgery (4w)", value=False)
        chk_prev_vte = W.Checkbox(description="Prior VTE", value=False)
        chk_hemo     = W.Checkbox(description="Hemoptysis", value=False)
        chk_cancer   = W.Checkbox(description="Active malignancy", value=False)
        chk_estrogen = W.Checkbox(description="Estrogen use (PERC)", value=False)
        chk_trauma   = W.Checkbox(description="Recent trauma (4w, PERC)", value=False)
        chk_unilat   = W.Checkbox(description="Unilateral leg swelling (PERC)", value=False)

        # --- Wells-DVT flags ---
        dvt_cancer   = W.Checkbox(description="Active cancer", value=False)
        dvt_paresis  = W.Checkbox(description="Paresis / plaster cast", value=False)
        dvt_bed_surg = W.Checkbox(description="Bedridden ≥3d / surgery ≤12w", value=False)
        dvt_tender   = W.Checkbox(description="Deep vein tenderness", value=False)
        dvt_entire   = W.Checkbox(description="Entire leg swollen", value=False)
        dvt_calf3    = W.Checkbox(description="Calf swelling >3 cm", value=False)
        dvt_edema    = W.Checkbox(description="Pitting edema (symptomatic leg)", value=False)
        dvt_collat   = W.Checkbox(description="Collateral non-varicose", value=False)
        dvt_prev     = W.Checkbox(description="Previous DVT", value=False)
        dvt_alt_dx   = W.Checkbox(description="Alternative dx as likely (subtract)", value=False)

        # --- HL7 paste area ---
        hl7_in = W.Textarea(
            value='''OBX|1|NM|BILIRUBIN TOTAL||36|umol/L|||N||F|||20250820090000
OBX|2|NM|INR||1.9|||N||F|||20250820090000
OBX|3|NM|ALBUMIN||28|g/L|||N||F|||20250820090000
OBX|4|NM|D-DIMER||780|ug/L|||H||F|||20250820090000
OBX|5|NM|TROPONIN T||78|ng/L|||H||F|||20250820090000
OBX|6|NM|TROPONIN T||18|ng/L|||N||F|||20250820050000''',
            description="HL7 paste", layout=W.Layout(width="100%", height="140px")
        )

        btn_apply_hl7 = W.Button(description="Apply HL7 → Labs JSON")
        btn_compute   = W.Button(description="Compute + Log")
        out = W.Output()

        def on_apply(_):
            with out:
                out.clear_output()
                try:
                    labs = json.loads(labs_in.value or "{}")
                except Exception as e:
                    print("[labs parse error]", e)
                    return
                parsed = parse_hl7_labs(hl7_in.value or "")
                # Merge core others
                oth = parsed.get("others") or {}
                for k in ["bilirubin","inr","albumin"]:
                    if oth.get(k) is not None:
                        labs[k] = oth[k]
                # If available, set D-dimer (take newest) when missing
                dd = parsed.get("d_dimer") or []
                if dd and "d_dimer" not in labs:
                    dd_sorted = sorted(dd, key=lambda t: (t[0] or 0))
                    _, v, u = dd_sorted[-1]
                    labs["d_dimer"] = v
                    labs["d_dimer_unit"] = u
                # Always (re)write troponin_series if present
                ts = parsed.get("troponin") or []
                if ts:
                    series = []
                    for t, v, u in ts:
                        series.append({
                            "time": (t.isoformat()+"Z") if t else None,
                            "value": v, "unit": u or "ng/L"
                        })
                    labs["troponin_series"] = series
                labs_in.value = json.dumps(labs, ensure_ascii=False)
                print("HL7 merged →", {k: labs.get(k) for k in ["bilirubin","inr","albumin","d_dimer","d_dimer_unit"]})

        def on_compute(_):
            with out:
                out.clear_output()
                try:
                    vitals = json.loads(vitals_in.value or "{}")
                    labs   = json.loads(labs_in.value or "{}")
                    ctx    = json.loads(ctx_in.value or "{}")
                except Exception as e:
                    print("[parse error]", e)
                    return

                # HEART sub-scores from dropdowns
                vitals["heart_history"] = int(heart_hist.value)
                vitals["heart_ecg"]     = int(heart_ecg.value)
                vitals["heart_risk"]    = int(heart_risk.value)

                # PE / PERC + DVT flags
                ctx.update({
                    "pe_most_likely": chk_pe_most.value,
                    "dvt_signs": chk_dvt_signs.value,
                    "immobilized": chk_immob.value,
                    "recent_surgery_4w": chk_immob.value,
                    "prev_vte": chk_prev_vte.value,
                    "hemoptysis": chk_hemo.value,
                    "cancer_active": chk_cancer.value or ctx.get("cancer_active", False),
                    "estrogen_use": chk_estrogen.value,
                    "recent_trauma_4w": chk_trauma.value,
                    "unilateral_leg_swelling": chk_unilat.value,
                    # Wells-DVT details
                    "paresis": dvt_paresis.value,
                    "plaster_cast": dvt_paresis.value,
                    "bedridden_3d": dvt_bed_surg.value,
                    "surgery_12w": dvt_bed_surg.value,
                    "deep_vein_tenderness": dvt_tender.value,
                    "entire_leg_swollen": dvt_entire.value,
                    "calf_swelling_gt3cm": dvt_calf3.value,
                    "pitting_edema": dvt_edema.value,
                    "collateral_nonvaricose": dvt_collat.value,
                    "prev_dvt": dvt_prev.value,
                    "alt_dx_as_likely": dvt_alt_dx.value,
                })

                bundle = phase2_bundle(vitals, labs, ctx)

                # ML risk (only if bridge loaded)
                if "predict_one" in globals() and "FEAT" in globals():
                    feats = FEAT
                    row = {f: labs.get(f, vitals.get(f, ctx.get(f, 0.0))) for f in feats}
                    res = predict_one(row)
                    bundle["one_liners"].append(
                        f"ML risk p={res['p']:.3f} → {'ALERT' if res['y'] else 'ok'} (thr={res['thr']:.3f})"
                    )
                    try:
                        _append_event({"type": "ml_score", "keys": list(row.keys()), "res": res})
                    except Exception:
                        pass
                else:
                    bundle["one_liners"].append("ML risk: (bridge not loaded)")

                _append_event({"type":"phase2_bundle","bundle": bundle})
                print("\n".join(bundle["one_liners"]))
                print("\nLogged to:", EVENT_LOG_PATH)

        btn_apply_hl7.on_click(on_apply)
        btn_compute.on_click(on_compute)

        display(W.VBox([
            W.HTML("<b>Phase-2 calculators — optimized UI (PE/DVT/PERC, PESI/sPESI, SIRS/Sepsis3, HEART, GBS, Child-Pugh)</b>"),
            W.HBox([vitals_in, labs_in]),
            W.HBox([heart_hist, heart_ecg, heart_risk]),
            W.HTML("<b>PE / PERC flags</b>"),
            W.HBox([chk_pe_most, chk_dvt_signs, chk_immob, chk_prev_vte, chk_hemo, chk_cancer]),
            W.HBox([chk_estrogen, chk_trauma, chk_unilat]),
            W.HTML("<b>Wells-DVT flags</b>"),
            W.HBox([dvt_cancer, dvt_paresis, dvt_bed_surg, dvt_tender, dvt_entire]),
            W.HBox([dvt_calf3, dvt_edema, dvt_collat, dvt_prev, dvt_alt_dx]),
            ctx_in,
            W.HBox([btn_compute, btn_apply_hl7]),
            W.HTML("<b>HL7 Labs (Trop / D-Dimer / INR / Bili / Albumin)</b>"),
            hl7_in,
            out
        ]))
    except Exception as e:
        print("Phase-2 UI unavailable:", e)
else:
    # Text-mode smoke test (runs if RUN_UI=False)
    vit = {"rr":24,"sbp":95,"gcs":14,"hr":120,"temp":38.6,"avpu":"V","map":65,"age":68,"sex":"M","sao2":97,
           "heart_history":0,"heart_ecg":0,"heart_risk":0}
    lab = {"d_dimer":780,"d_dimer_unit":"μg/L FEU",
           "troponin_series":[{"time":"2025-08-20T05:00:00Z","value":18,"unit":"ng/L"},{"time":"2025-08-20T09:00:00Z","value":78,"unit":"ng/L"}],
           "creatinine":1.8,"platelets":95,"albumin":2.8,"calcium":7.9,"na":138,"cl":105,"hco3":20,"bilirubin":2.1,"inr":1.9}
    ctx = {"pregnant": True, "suspected_infection": True, "vasopressors": False, "pe_most_likely": True, "dvt_signs": False,
           "ascites":"mild","encephalopathy":"1-2"}
    b = phase2_bundle(vit, lab, ctx)
    _append_event({"type":"phase2_bundle_smoke","bundle": b})
    print("\n".join(b["one_liners"]))
    print("Logged to:", CONFIG.get("EVENT_LOG_PATH", "/mnt/data/event_log.jsonl"))


In [97]:
def gate_engine_with_capacity(fd, CONFIG):
    return merge_gatepacks(gate_engine(fd, CONFIG), capacity_gatepack(fd, CONFIG))
fd = {"suspected_condition":"respiratory","cap_internal_medicine_icu":0.0,"ems_triage_code":1,"admit_decision":"ICU","Triage":"Orange"}
pack = capacity_gatepack(fd, CONFIG)
print(pack["gates"], "|", pack["next_action"])
# Expect ICU_BLOCKED, ED_BOARDING_RISK, ROUTE_PLAN_ED_TO_ICU, ALERT_* , TRANSFER_BLOCK_CAPACITY


['ICU_BLOCKED', 'ED_BOARDING_RISK', 'ROUTE_PLAN_ED_TO_ICU', 'ALERT_NURSE', 'ALERT_DOC', 'ALERT_ATTENDING', 'TRANSFER_BLOCK_CAPACITY'] | No ICU capacity — board in ED, activate bed manager now.


In [98]:
# Force-sync all "Context JSON" textareas and the reference used by YEARS
import json, ipywidgets as W, gc

def _update_all_context_widgets(flag=True):
    new_val = None
    # If a ctx_in exists in this scope, start from it
    try:
        d = json.loads(ctx_in.value or "{}")
    except Exception:
        d = {}
    d["pregnant"] = bool(flag)
    new_val = json.dumps(d, ensure_ascii=False)

    # Update any Textarea whose description starts with "Context JSON"
    n = 0
    for obj in gc.get_objects():
        try:
            if isinstance(obj, W.Textarea) and (obj.description or "").startswith("Context JSON"):
                obj.value = new_val
                n += 1
        except Exception:
            pass
    print(f"Updated {n} Context JSON widget(s) →", new_val)

    # Make sure the YEARS cell reads the same widget reference
    global ctx_src  # used inside the YEARS cell handler
    try:
        ctx_src = ctx_in
        print("ctx_src → ctx_in (bound)")
    except NameError:
        print("ctx_in not in scope; YEARS will still read its own ctx_src if present.")

_update_all_context_widgets(flag=True)


Updated 2 Context JSON widget(s) → {"pregnant": true, "suspected_infection": true, "vasopressors": false, "pe_most_likely": true, "dvt_signs": false, "ascites": "mild", "encephalopathy": "1-2"}
ctx_src → ctx_in (bound)


In [99]:

# === Pregnancy-adapted YEARS pathway (drop-in) ===
# Uses the Phase-2 panel's Context/Labs if available; otherwise exposes its own minimal text areas.
# Logs to CONFIG['EVENT_LOG_PATH'] as {"type": "pregnancy_years", ...}

from datetime import datetime, timezone
import json

def assess_pregnancy_years(ctx, labs):
    # Preg-adapted YEARS: three items — DVT signs, hemoptysis, 'PE most likely'.
    # If none present → D-dimer threshold 1000 μg/L FEU; else threshold 500 μg/L FEU.
    preg = bool(ctx.get("pregnant", False))
    items = {
        "dvt_signs": bool(ctx.get("dvt_signs", False)),
        "hemoptysis": bool(ctx.get("hemoptysis", False)),
        "pe_most_likely": bool(ctx.get("pe_most_likely", False)),
    }
    count = sum(int(v) for v in items.values())
    d_val = None
    for k in ("d_dimer","ddimer","d-dimer","d_dimer_feu_ug_l"):
        if k in labs and labs[k] not in (None, ""):
            try:
                d_val = float(str(labs[k]).replace(",", "."))
                break
            except Exception:
                pass
    d_unit = (labs.get("d_dimer_unit") or "μg/L FEU").lower()
    if d_val is None:
        return {"applies": preg, "needs_imaging": True, "reason": "missing D-dimer", "items": items, "items_count": count}
    d_ug = float(d_val)*1000.0 if "mg/l" in d_unit else float(d_val)
    thr = 1000.0 if count==0 else 500.0
    ruleout = bool(d_ug < thr)
    return {
        "applies": preg, "items": items, "items_count": count,
        "d_dimer_ug_l": d_ug, "threshold_ug_l": thr, "rule_out": ruleout,
        "note": "If DVT signs present, compression ultrasound is recommended before applying YEARS."
    }

# UI that reuses Phase-2 widget inputs if present (vitals_in / labs_in / ctx_in).
try:
    _probe = vitals_in  # noqa: F401
    _has_phase2_ui = True
except NameError:
    _has_phase2_ui = False

if CONFIG.get("RUN_UI", False):
    try:
        import ipywidgets as W
        if _has_phase2_ui:
            labs_src = labs_in
            ctx_src  = ctx_in
            src_note_ctx = W.HTML('<i>Using Context JSON from Phase-2 panel</i>')
            src_note_labs = W.HTML('<i>Using Labs JSON from Phase-2 panel</i>')
        else:
            labs_src = W.Textarea(value='{"d_dimer":780,"d_dimer_unit":"μg/L FEU"}', description="Labs JSON", layout=W.Layout(width="100%", height="80px"))
            ctx_src  = W.Textarea(value='{"pregnant": true, "pe_most_likely": true, "dvt_signs": false, "hemoptysis": false}', description="Context JSON", layout=W.Layout(width="100%", height="80px"))
            src_note_ctx = src_note_labs = W.HTML('')
        out = W.Output()
        btn = W.Button(description="Compute Pregnancy YEARS + Log")
        warn = W.HTML("<small><b>Note:</b> PERC is not validated in pregnancy; use the YEARS pathway below.</small>")

        def _on_click(_):
            with out:
                out.clear_output()
                try:
                    labs = json.loads(labs_src.value if hasattr(labs_src, 'value') else labs_src)
                    ctx  = json.loads(ctx_src.value if hasattr(ctx_src, 'value') else ctx_src)
                except Exception as e:
                    print("[parse error]", e); return
                res = assess_pregnancy_years(ctx, labs)
                if not res.get("applies", False):
                    print("Pregnancy YEARS: not applicable (pregnant flag is false).")
                else:
                    msg = f"Pregnancy YEARS: items={res['items_count']} → D-dimer thr {int(res['threshold_ug_l'])} μg/L; value {int(res['d_dimer_ug_l'])} → "
                    msg += ("RULE-OUT" if res["rule_out"] else "IMAGING")
                    print(msg)
                try:
                    _append_event({"type":"pregnancy_years", "result": res})
                    print("Logged to:", CONFIG.get("EVENT_LOG_PATH"))
                except Exception as e:
                    print("[log error]", e)

        btn.on_click(_on_click)

        display(W.VBox([
            W.HTML("<b>Pregnancy-adapted YEARS pathway</b>"),
            warn,
            src_note_ctx, src_note_labs,
            btn, out
        ]))
    except Exception as e:
        print("YEARS UI unavailable:", e)

# Text-mode smoke if RUN_UI is off
if not CONFIG.get("RUN_UI", False):
    labs = {"d_dimer": 780, "d_dimer_unit": "μg/L FEU"}
    ctx = {"pregnant": True, "pe_most_likely": True, "dvt_signs": False, "hemoptysis": False}
    res = assess_pregnancy_years(ctx, labs)
    msg = f"Pregnancy YEARS (text-mode): items={res.get('items_count',0)}; thr={int(res.get('threshold_ug_l',0))} μg/L; value={int(res.get('d_dimer_ug_l',0))} → "
    msg += ("RULE-OUT" if res.get("rule_out") else "IMAGING")
    print(msg)
    try:
        _append_event({"type":"pregnancy_years_smoke", "result": res})
        print("Logged to:", CONFIG.get("EVENT_LOG_PATH"))
    except Exception as e:
        print("[log error]", e)


In [100]:
# === Safe tail of EVENT_LOG_PATH (optional) ===
import os, subprocess, shlex
from pathlib import Path
EVENT_LOG_PATH = CONFIG.get("EVENT_LOG_PATH", "/kaggle/working/event_log.jsonl")
if Path(EVENT_LOG_PATH).exists():
    try:
        subprocess.run(["tail","-n","20",EVENT_LOG_PATH], check=False)
    except Exception as e:
        print("tail failed (ignored):", e)
else:
    print("No event log yet (will be created on first write).")

No event log yet (will be created on first write).


In [101]:

# === Medication Plan Import + Checks (inline, no sidecars) ===
from __future__ import annotations
from typing import Dict, Any, List, Optional, Tuple
from pathlib import Path
from datetime import datetime, timezone
import json, re

EVENT_LOG = Path(CONFIG["EVENT_LOG_PATH"])

def _append_event(ev: Dict[str, Any]):
    EVENT_LOG.parent.mkdir(parents=True, exist_ok=True)
    EVENT_LOG.touch(exist_ok=True)
    ev = {"ts": datetime.now(timezone.utc).isoformat(), **ev}
    with EVENT_LOG.open("a") as fp:
        fp.write(json.dumps(ev, ensure_ascii=False) + "\n")

# Minimal ATC/name map for demo (extendable)
_ATC_MAP = {
    "amoxicillin": {"atc": "J01CA04", "class": "penicillin"},
    "ibuprofen": {"atc": "M01AE01", "class": "nsaid"},
    "warfarin": {"atc": "B01AA03", "class": "coumarin"},
    "ramipril": {"atc": "C09AA05", "class": "ace"},
    "spironolacton": {"atc": "C03DA01", "class": "aldosterone_antagonist"},
    "metoprolol": {"atc": "C07AB02", "class": "beta_blocker"},
    "simvastatin": {"atc": "C10AA01", "class": "statin"},
    "clarithromycin": {"atc": "J01FA09", "class": "macrolide"},
    "azithromycin": {"atc": "J01FA10", "class": "macrolide"},
}

# Demo interaction rules (replace via MED_RULES_PATH for your own rules)
_DEMO_RULES = [
    {"a": "ibuprofen", "b": "warfarin", "severity": "major", "message": "Bleeding risk (NSAID + warfarin)"},
    {"a": "ramipril", "b": "spironolacton", "severity": "moderate", "message": "Hyperkalemia risk (ACE + aldosterone antagonist)"},
    {"a": "amoxicillin", "b": "warfarin", "severity": "moderate", "message": "Antibiotic may potentiate warfarin effect"},
    {"a": "clarithromycin", "b": "simvastatin", "severity": "major", "message": "Rhabdomyolysis risk (CYP3A4 inhibition)"},
]

_ROUTES = ["p.o.", "po", "i.v.", "iv", "i.m.", "im", "s.c.", "sc", "inhalativ", "topisch", "nasal", "otic", "ophthalmic"]
_FREQ_WORDS = ["morgens", "mittags", "abends", "nachts"]
_FREQ_PAT = re.compile(r"\b(\d+)[-/.](\d+)[-/.](\d+)\b")
_DOSE_PAT = re.compile(r"(\d+(?:[.,]\d+)?)\s*(mg|g|mcg|µg|ml|IE|Einheiten)\b", flags=re.I)

def _normalize_name(s: str) -> str:
    s = s.strip().lower()
    s = s.replace("ä","ae").replace("ö","oe").replace("ü","ue").replace("ß","ss")
    return re.sub(r"[^a-z0-9]+", " ", s).strip()

def _map_to_atc(name_norm: str) -> Dict[str, Any]:
    for key, meta in _ATC_MAP.items():
        if key in name_norm:
            return {"name_norm": key, **meta}
    return {"name_norm": name_norm, "atc": None, "class": None}

def parse_med_line(line: str) -> Optional[Dict[str, Any]]:
    raw = line.strip()
    if not raw or raw.startswith("#"):
        return None
    name = raw.split(",")[0].split("  ")[0]  # up to comma or double spaces
    name_norm = _normalize_name(name)
    dose_val, dose_unit = None, None
    m = _DOSE_PAT.search(raw)
    if m:
        dose_val = float(m.group(1).replace(",", "."))
        dose_unit = m.group(2).lower()
    freq = None
    m = _FREQ_PAT.search(raw)
    if m:
        freq = f"{m.group(1)}-{m.group(2)}-{m.group(3)}"
    else:
        words = [w for w in _FREQ_WORDS if w in raw.lower()]
        if words:
            freq = ",".join(words)
    route = None
    for r in _ROUTES:
        if r in raw.lower():
            route = r
            break
    prn = bool(re.search(r"\b(prn|bei bedarf)\b", raw, flags=re.I))
    atc_meta = _map_to_atc(name_norm)
    return {
        "raw": raw,
        "name": name.strip(),
        "name_norm": atc_meta["name_norm"],
        "atc": atc_meta["atc"],
        "class": atc_meta["class"],
        "dose_value": dose_val,
        "dose_unit": dose_unit,
        "frequency": freq,
        "route": route,
        "prn": prn,
    }

def parse_med_text(text: str) -> List[Dict[str, Any]]:
    meds = []
    for line in text.splitlines():
        rec = parse_med_line(line)
        if rec:
            meds.append(rec)
    return meds

def load_rules(path: str) -> List[Dict[str, Any]]:
    p = Path(path)
    if p.exists():
        try:
            js = json.loads(p.read_text())
            if isinstance(js, list):
                return js
        except Exception:
            pass
    return list(_DEMO_RULES)

def load_allergies(path: str) -> List[str]:
    p = Path(path)
    if p.exists():
        try:
            js = json.loads(p.read_text())
            if isinstance(js, dict) and "allergies" in js and isinstance(js["allergies"], list):
                return [str(x) for x in js["allergies"]]
        except Exception:
            pass
    return []

def check_interactions(meds: List[Dict[str, Any]], rules: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out = []
    names = [m["name_norm"] for m in meds]
    for r in rules:
        a, b = r["a"], r["b"]
        if (a in names and b in names) or (b in names and a in names):
            out.append({"type": "med_interaction_warning", **r})
    return out

def check_allergies(meds: List[Dict[str, Any]], allergy_terms: List[str]) -> List[Dict[str, Any]]:
    out = []
    terms = [t.lower() for t in allergy_terms]
    for m in meds:
        # direct name match
        if any(t in m["name"].lower() for t in terms):
            out.append({"type": "med_allergy_warning", "med": m["name"], "match": "name"})
            continue
        # conservative penicillin class heuristic
        if any("penicillin" in t for t in terms):
            if m["class"] == "penicillin" or m["name"].lower().endswith("cillin"):
                out.append({"type": "med_allergy_warning", "med": m["name"], "match": "class_penicillin"})
    return out

def ocr_file(path: str) -> Optional[str]:
    p = Path(path)
    if not p.exists():
        return None
    try:
        from PIL import Image
        import pytesseract
        if p.suffix.lower() in [".png",".jpg",".jpeg",".tif",".tiff"]:
            return pytesseract.image_to_string(Image.open(p))
        if p.suffix.lower() == ".pdf":
            try:
                from pdf2image import convert_from_path
                pages = convert_from_path(str(p))
                text = ""
                for img in pages:
                    text += pytesseract.image_to_string(img) + "\n"
                return text
            except Exception:
                return None
    except Exception:
        return None
    return None

def emit_meds_events(patient_id: str, meds: List[Dict[str, Any]], warnings: List[Dict[str, Any]]):
    _append_event({"type": "med_plan_import", "patient_id": patient_id, "n_meds": len(meds)})
    for m in meds:
        _append_event({"type": "med_entry", "patient_id": patient_id, **m})
    for w in warnings:
        _append_event({**w, "patient_id": patient_id})

if CONFIG.get("RUN_MEDS"):
    print("Medication import/checks enabled. Use the UI panel (if RUN_UI=True) or call the functions above.")
else:
    print("Deferred… set CONFIG['RUN_MEDS']=True to enable medication import & checks.")

# Optional UI
if CONFIG.get("RUN_UI") and CONFIG.get("RUN_MEDS"):
    try:
        import ipywidgets as W
        import pandas as pd
        pid = W.Text(description="Patient ID", placeholder="e.g., UKE-12345")
        path = W.Text(description="Scan path", placeholder="/mnt/data/scan.pdf (optional)")
        ocr_btn = W.Button(description="Run OCR")
        ta = W.Textarea(description="Plan text", layout=W.Layout(width="100%", height="180px"))
        parse_btn = W.Button(description="Parse & Check", button_style="primary")
        out = W.Output()

        def on_ocr(_):
            with out:
                out.clear_output()
                txt = ocr_file(path.value.strip())
                if txt:
                    ta.value = txt
                    print("OCR complete.")
                else:
                    print("OCR unavailable or failed. Paste the text instead.")

        def on_parse(_):
            with out:
                out.clear_output()
                text = ta.value.strip()
                if not text:
                    print("No text provided."); return
                meds = parse_med_text(text)
                rules = load_rules(CONFIG["MED_RULES_PATH"])
                allergies = load_allergies(CONFIG["ALLERGIES_PATH"])
                warnings = check_interactions(meds, rules) + check_allergies(meds, allergies)
                emit_meds_events(pid.value or "DEMO", meds, warnings)
                if meds:
                    display(pd.DataFrame(meds))
                else:
                    print("No medications parsed.")
                if warnings:
                    print("\nWarnings:")
                    display(pd.DataFrame(warnings))
                else:
                    print("\nNo interaction/allergy warnings.")

        ocr_btn.on_click(on_ocr)
        parse_btn.on_click(on_parse)
        display(W.VBox([pid, path, W.HBox([ocr_btn, parse_btn]), ta, out]))
    except Exception as e:
        print("Meds UI unavailable:", e)


Medication import/checks enabled. Use the UI panel (if RUN_UI=True) or call the functions above.


In [102]:
# === Gate-pack merger (pure; TTL=max; order-preserving dedup) ===
from typing import Dict, Any

def _merge_ttl_max(a: Dict[str,int]|None, b: Dict[str,int]|None) -> Dict[str,int]:
    out = dict(a or {})
    for k, v in (b or {}).items():
        try:
            out[k] = max(out.get(k, 0), int(v))
        except Exception:
            out[k] = out.get(k, 0)
    return out

def merge_gatepacks(*packs: Dict[str, Any]) -> Dict[str, Any]:
    out: Dict[str, Any] = {"gates": [], "priority": 0, "next_action": "", "explain": [], "ttl": {}}
    for p in packs:
        if not p: 
            continue
        out["gates"].extend(p.get("gates", []))
        out["priority"] = max(out["priority"], int(p.get("priority", 0)))
        if p.get("next_action"):
            out["next_action"] = p["next_action"]
        if p.get("explain"):
            out["explain"].extend(p["explain"])
        out["ttl"] = _merge_ttl_max(out["ttl"], p.get("ttl"))
    # order-preserving dedup
    seen, dedup = set(), []
    for g in out["gates"]:
        if g not in seen:
            seen.add(g); dedup.append(g)
    out["gates"] = dedup
    return out


In [103]:
# === Gate-engine (composes packs; ICU & BGA fallback if packs absent) ===
from typing import Dict, Any

def _fallback_pack_icu(fd: Dict[str,Any], CONFIG: Dict[str,Any]) -> Dict[str,Any]:
    # thresholds (do not mutate CONFIG)
    th_ok     = float(CONFIG.get("TH_ICU_CAP_OK", 0.15))     # ≥15% = OK
    cap       = fd.get("cap_internal_medicine_icu")
    need_icu  = str(fd.get("admit_decision","")).upper() == "ICU"
    blocked   = (cap is not None) and (float(cap) <= 0.0)
    tight     = (cap is not None) and (0.0 < float(cap) < th_ok)
    gates, explain = [], []
    if need_icu and (blocked or tight):
        gates += ["ROUTE_PLAN_ED_TO_ICU"]
        if blocked:
            gates += ["ICU_BLOCKED","ED_BOARDING_RISK","TRANSFER_BLOCK_CAPACITY"]
            explain.append("No ICU capacity — board in ED, activate bed manager now.")
        elif tight:
            gates += ["ICU_TIGHT"]
            explain.append("ICU capacity tight — pre-alert bed manager and accepting team.")
        # alerts for capacity issue
        gates += ["ALERT_NURSE","ALERT_DOC"]
    if not gates:
        return {"gates": [], "priority": 0, "next_action": "", "explain": [], "ttl": {}}
    return {
        "gates": gates,
        "priority": 1,
        "next_action": explain[0] if explain else "",
        "explain": explain,
        "ttl": {"ALERT_NURSE":300, "ALERT_DOC":300}
    }

def _fallback_pack_bga(fd: Dict[str,Any], CONFIG: Dict[str,Any]) -> Dict[str,Any]:
    # thresholds from your ops policy
    pH  = fd.get("vbg_pH")
    K   = fd.get("vbg_K")
    Na  = fd.get("vbg_Na")
    Lac = fd.get("vbg_lactate")
    gates, explain, ttl = [], [], {}
    # pH
    if pH is not None:
        if float(pH) > 7.55:
            gates.append("BGA_ALKALOSIS_CRITICAL"); explain.append(f"pH {pH} > 7.55")
        elif float(pH) >= 7.5:
            gates.append("BGA_ALKALOSIS_HIGH"); explain.append(f"pH {pH} ≥ 7.5")
    # Potassium
    if K is not None:
        kf = float(K)
        if kf >= 6.5: gates.append("BGA_HYPERKALAEMIA_CRITICAL"); explain.append(f"K⁺ {kf} ≥ 6.5 mmol/L")
        elif kf >= 6.0: gates.append("BGA_HYPERKALAEMIA_HIGH"); explain.append(f"K⁺ {kf} ≥ 6.0 mmol/L")
        elif kf < 2.5: gates.append("BGA_HYPOKALAEMIA_CRITICAL"); explain.append(f"K⁺ {kf} < 2.5 mmol/L")
    # Sodium
    if Na is not None:
        naf = float(Na)
        if naf <= 125: gates.append("BGA_DYSNATRAEMIA_CRITICAL"); explain.append(f"Na⁺ {naf} ≤ 125 mmol/L")
        elif (naf <= 129) or (naf >= 145): gates.append("BGA_DYSNATRAEMIA"); explain.append(f"Na⁺ {naf} outside 130–144 mmol/L")
    # Lactate (typical critical)
    if Lac is not None and float(Lac) >= 4.0:
        gates.append("BGA_LACTATE_CRITICAL"); explain.append(f"Lactate {Lac} ≥ 4.0 mmol/L")
    # escalate alerts for any critical
    if any(g.endswith("CRITICAL") for g in gates):
        gates += ["ALERT_NURSE","ALERT_DOC","ALERT_ATTENDING"]
        ttl.update({"ALERT_NURSE":300,"ALERT_DOC":300,"ALERT_ATTENDING":300})
    if not gates:
        return {"gates": [], "priority": 0, "next_action": "", "explain": [], "ttl": {}}
    return {
        "gates": gates,
        "priority": 2 if any(g.endswith("CRITICAL") for g in gates) else 1,
        "next_action": "Address critical VBG derangements now." if any(g.endswith("CRITICAL") for g in gates) else "",
        "explain": explain,
        "ttl": ttl
    }

def gate_engine(fd: Dict[str,Any], CONFIG: Dict[str,Any]) -> Dict[str,Any]:
    """
    Compose gate-packs if present; otherwise use ICU+BGA fallbacks to guarantee safety signals.
    No I/O. Pure function.
    """
    packs = []
    # Prefer your real pack producers if they’re defined in the notebook/module
    for name in [
        # add your real producers here if present, e.g.:
        "ops_capacity_gatepack",           # ICU/boarding pack
        "bga_gatepack",                    # VBG/electrolytes pack
        "abdomen_guardrails_gatepack",     # imaging + labs policy
        "overdue_gatepack",                # last_seen & vitals overdues
        "troponin_policy_gatepack",        # troponin timing/delta policy
    ]:
        fn = globals().get(name)
        if callable(fn):
            try:
                packs.append(fn(fd, CONFIG))
            except Exception as e:
                # non-fatal: continue composing with other packs
                packs.append({"gates":[], "priority":0, "next_action":"", "explain":[f"{name} failed: {e}"], "ttl":{}})

    # If key packs are missing, add safe fallbacks
    if not any("ICU_" in g for p in packs for g in p.get("gates", [])):
        packs.append(_fallback_pack_icu(fd, CONFIG))
    if not any(g.startswith("BGA_") for p in packs for g in p.get("gates", [])):
        packs.append(_fallback_pack_bga(fd, CONFIG))

    # Merge all packs
    return merge_gatepacks(*packs)


In [104]:
# === Failure Triage Harness (gate engine + Phase-2 split) ===
import sys, types, inspect
from datetime import datetime

def _have(*names):
    return {n: (n in globals()) for n in names}

def _where(fn_name):
    obj = globals().get(fn_name, None)
    if not callable(obj): return None
    try:
        src = inspect.getsourcefile(obj) or inspect.getfile(obj)
        ln  = obj.__code__.co_firstlineno
        return f"{src}:{ln}"
    except Exception:
        return "<dynamic>"

print(f"PY {sys.version.split()[0]}")
print("PRESENT:", _have("merge_gatepacks","gate_engine","_pick","_bool","_safe_round",
                       "build_labs_norm","compute_phase2_scores","phase2_bundle",
                       "parse_hl7_all","parse_hl7_labs"))

# Alias tolerance for qSOFA func name drift
if "calc_qsoFA" not in globals() and "calc_qsofa" in globals():
    calc_qsoFA = calc_qsofa
elif "calc_qsoFA" in globals() and "calc_qsofa" not in globals():
    calc_qsofa = calc_qsoFA

print("LOC:", {n:_where(n) for n in ["merge_gatepacks","gate_engine","build_labs_norm",
                                     "compute_phase2_scores","phase2_bundle"]})

def _probe(title, fn):
    try:
        r = fn()
        print(f"{title}: OK")
        return True, r
    except Exception as e:
        et = type(e).__name__
        print(f"{title}: FAIL -> {et}: {e}")
        return False, e

# 1) TTL merge sanity
def _t1():
    p1 = {"gates":["ALERT_NURSE"], "priority":0, "next_action":"", "explain":[], "ttl":{"ALERT_NURSE":120}}
    p2 = {"gates":["ALERT_NURSE"], "priority":0, "next_action":"", "explain":[], "ttl":{"ALERT_NURSE":300}}
    m  = merge_gatepacks(p1, p2)
    assert m["ttl"].get("ALERT_NURSE")==300
    return m
_probe("TTL_MAX_OK", _t1)

# 2) Gate engine probe (no HL7 needed)
def _t2():
    fd = {"Triage":"Orange","admit_decision":"ICU","suspected_condition":"respiratory",
          "cap_internal_medicine_icu":0.0,"ems_triage_code":1,"vbg_pH":7.56,"vbg_K":6.6}
    out = gate_engine(fd, CONFIG)
    gs = set(out.get("gates",[]))
    assert {"ALERT_NURSE","ALERT_DOC"}.intersection(gs)
    return sorted(list(gs))[:8]
_probe("GATES_WIRED_OK", _t2)

# 3) Phase-2 bundle probe (uses canonical HL7 if available; otherwise minimal labs)
def _t3():
    if "parse_hl7_all" in globals():
        hl7 = ("OBX|1|NM|D-DIMER^D-Dimer FEU||2000|µg/L FEU|||F\n"
               "OBX|2|NM|TROP^Troponin T (hs)||70|ng/L|||F\n"
               "OBX|3|NM|BILITOT^Bilirubin Total||2.3|mg/dL|||F\n"
               "OBX|4|NM|ALB^Albumin||3.6|g/dL|||F\n")
        labs = parse_hl7_all(hl7)
    else:
        labs = {"lab_ddimer_mgL_FEU":2.0,"lab_bili_total_umolL":2.3*17.104,"lab_albumin_gL":36.0}
    vit = {"sbp":110,"hr":95,"rr":18,"temp":37.1,"gcs":15}
    out = phase2_bundle(vit, labs, {"age":62})
    assert "one_liners" in out and "scores" in out
    return out["scores"].get("qsofa"), out["one_liners"][:2]
_probe("PHASE2_BUNDLE_SMOKE_OK", _t3)

print("HINTS:")
print("- NameError merge_gatepacks/gate_engine ⇒ run your gate packs/engine cell or add the merger shim above.")
print("- NameError _pick/_bool/_safe_round ⇒ run Calc utils cell (defines those three).")
print("- calc_qsoFA vs calc_qsofa mismatch ⇒ alias applied above; make it permanent later.")
print("- NameError parse_hl7_all ⇒ run unified HL7 cell; bundle fallback used here until you do.")


PY 3.11.13
PRESENT: {'merge_gatepacks': True, 'gate_engine': True, '_pick': True, '_bool': True, '_safe_round': True, 'build_labs_norm': True, 'compute_phase2_scores': True, 'phase2_bundle': True, 'parse_hl7_all': False, 'parse_hl7_labs': False}
LOC: {'merge_gatepacks': '/tmp/ipykernel_36/1608344827.py:13', 'gate_engine': '/tmp/ipykernel_36/2783023910.py:73', 'build_labs_norm': '/tmp/ipykernel_36/275384517.py:4', 'compute_phase2_scores': '/tmp/ipykernel_36/1671391004.py:4', 'phase2_bundle': '/tmp/ipykernel_36/1664930203.py:5'}
TTL_MAX_OK: OK
GATES_WIRED_OK: OK
PHASE2_BUNDLE_SMOKE_OK: OK
HINTS:
- NameError merge_gatepacks/gate_engine ⇒ run your gate packs/engine cell or add the merger shim above.
- NameError _pick/_bool/_safe_round ⇒ run Calc utils cell (defines those three).
- calc_qsoFA vs calc_qsofa mismatch ⇒ alias applied above; make it permanent later.
- NameError parse_hl7_all ⇒ run unified HL7 cell; bundle fallback used here until you do.


In [105]:
# Gate dedup + TTL max policy
p1 = {"gates":["ALERT_NURSE"], "priority":0, "next_action":"", "explain":[], "ttl":{"ALERT_NURSE":120}}
p2 = {"gates":["ALERT_NURSE"], "priority":0, "next_action":"", "explain":[], "ttl":{"ALERT_NURSE":300}}
m  = merge_gatepacks(p1, p2)
assert m["ttl"].get("ALERT_NURSE") == 300
print("TTL_MAX_OK")

# Gate pack smoke (ICU + BGA path still wired)
fd = {"suspected_condition":"respiratory","cap_internal_medicine_icu":0.0,"ems_triage_code":1,
      "admit_decision":"ICU","Triage":"Orange","vbg_pH":7.56, "vbg_K":6.6}
pack = gate_engine(fd, CONFIG)
assert {"ICU_BLOCKED","ED_BOARDING_RISK","ALERT_NURSE"}.issubset(set(pack["gates"]))
print("GATES_WIRED_OK")


TTL_MAX_OK
GATES_WIRED_OK


In [106]:
# --- Phase-1 quickcheck (overlay; runs only if you call it) ---
from pathlib import Path
import inspect, pandas as pd

def phase1_quickcheck(config) -> dict:
    CFG = dict(config)           # do NOT mutate CONFIG
    CFG["RUN_UI"] = False
    CFG["RUN_PIPELINE"] = False

    # Ensure CSV/dir semantics without side effects
    for k in ("QR_OUTPUT_DIR","EVENT_LOG_PATH","SOP_REGISTRY_PATH",
              "EQUIPMENT_STATUS_PATH","EQUIPMENT_MOVES_LOG_PATH"):
        p = Path(CFG[k]); (p.parent if p.suffix else p).mkdir(parents=True, exist_ok=True)

    # Import tracker from alias or globals
    try:
        from tracker_core import TrackerService, QRService, SOPRegistry  # alias shim if present
    except Exception:
        TrackerService, QRService, SOPRegistry = TrackerService, QRService, SOPRegistry  # use in-notebook classes

    # 1) Moves log writes (no None-indexing thanks to read() patch)
    t = TrackerService.from_config(CFG)
    t.log_move("__smoke__", "", "ZZ")
    moves_ok = Path(CFG["EQUIPMENT_MOVES_LOG_PATH"]).exists()

    # 2) SOP auto-pull surface (offline-safe summary)
    try:
        sop_ok = isinstance(refresh_sop_registry(CFG, base_url="http://127.0.0.1:9/offline"), dict)
    except Exception:
        sop_ok = False

    # 3) Adjustable vitals threshold present (split-aware; no regex)
    try:
        ui_src = ""
        if "run_ui_header_controls" in globals():
            ui_src = inspect.getsource(run_ui_header_controls)
        elif "run_ui" in globals():
            ui_src = inspect.getsource(run_ui)
        ui_ok = "Vitals overdue (min)" in ui_src
    except Exception:
        ui_ok = False

    return {"moves": moves_ok, "sop": sop_ok, "ui_vitals_threshold": ui_ok}

# Example opt-in usage:
# res = phase1_quickcheck(CONFIG); print(res); assert all(res.values()), res


In [107]:
# one-ping-per-episode (arrival-only)
from datetime import datetime, timezone

FLOW = dict(monitor_paused=False, episode=0, notified_ep=None, current_service="ED")

def orderly(event, dest=None, now=None):
    t = now or datetime.now(timezone.utc)
    if event == "PICKUP":
        FLOW["episode"] += 1
        FLOW["monitor_paused"] = True
        if dest: FLOW["current_service"] = dest
        FLOW["notified_ep"] = None        # reset notifier for this episode
        # no nurse ping here
    elif event == "DROPOFF_BACK_ED":
        FLOW["current_service"] = "ED"
        # send exactly one arrival ping per episode
        if FLOW.get("notified_ep") != FLOW["episode"]:
            print(f"[NURSE] {t:%H:%M}Z Patient returned to ED — please resume monitoring.")
            FLOW["notified_ep"] = FLOW["episode"]
    elif event == "RESUME":
        FLOW["monitor_paused"] = False    # optional: no ping
    else:
        pass

# demo:
# orderly("PICKUP", dest="MRI")
# orderly("DROPOFF_BACK_ED")
# orderly("DROPOFF_BACK_ED")  # (no second ping)
# orderly("RESUME")


In [108]:
# --- Post-run spot checks (no CONFIG mutation) ---
import inspect, re, pandas as pd
CFG = dict(CONFIG)

# 1) EquipmentRepository.read patch effective?
from tracker_core import TrackerService
t = TrackerService.from_config(CFG)
df = t.equipment_status()
print("EQUIP_READ_OK:", isinstance(df, pd.DataFrame), "cols:", list(df.columns)[:7])

# 2) Adjustable vitals threshold present in UI?
try:
    src = inspect.getsource(run_ui)
    has_vitals_input = ("Vitals overdue" in src) or bool(re.search(r"number_input\\(.*vitals", src, flags=re.I))
    print("UI_VITALS_INPUT_OK:", has_vitals_input)
except Exception as e:
    print("UI_VITALS_INPUT_OK:", False, "| reason:", e)


EQUIP_READ_OK: True cols: ['equip_id', 'name', 'location', 'status', 'last_seen', 'battery', 'confidence']
UI_VITALS_INPUT_OK: False | reason: missing ), unterminated subpattern at position 14


In [109]:
# --- Fixed verifier for adjustable vitals threshold (no CONFIG mutation) ---
import inspect, re

src_ui  = ""
src_hdr = ""
try:
    src_ui  = inspect.getsource(run_ui)
except Exception:
    pass
try:
    src_hdr = inspect.getsource(run_ui_header_controls)
except Exception:
    pass

combined = (src_ui or "") + "\n" + (src_hdr or "")

# Simple substring checks
has_label  = "Vitals overdue (min)" in combined
has_number = "number_input" in combined and ("vitals" in combined.lower())

# Corrected regex (match a number_input call whose argument list mentions 'vitals')
regex_ok = bool(re.search(r'number_input\([^)]*vitals', combined, flags=re.I))

ui_vitals_adjustable = has_label or (has_number and regex_ok)

print("UI_VITALS_INPUT_OK:", ui_vitals_adjustable)
print("Found in:", ("run_ui_header_controls" if "Vitals overdue (min)" in (src_hdr or "") else "run_ui") if ui_vitals_adjustable else "NOT FOUND")


UI_VITALS_INPUT_OK: True
Found in: run_ui_header_controls


In [110]:
# === ARTIFACTS: setup + PDF helper ===
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path
import textwrap, json

OUT = Path("/kaggle/working")
(OUT / "tests").mkdir(parents=True, exist_ok=True)
NOW = datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")

def add_pdf_page(pdf, title, paragraphs, width=100, footer=""):
    """Minimal PDF page writer (A4). Safe to re-run."""
    fig = plt.figure(figsize=(8.27, 11.69), dpi=100)
    fig.subplots_adjust(left=0.07, right=0.93, top=0.92, bottom=0.07)
    plt.text(0.5, 0.965, title, ha="center", va="top", fontsize=16, fontweight="bold")
    y, line_h = 0.93, 0.022
    for p in paragraphs:
        for ln in textwrap.fill(p, width=width, replace_whitespace=False).split("\n"):
            if y < 0.08 + 2*line_h:
                if footer: plt.text(0.5, 0.045, footer, ha="center", va="center", fontsize=8, alpha=0.7)
                pdf.savefig(fig); plt.close(fig)
                fig = plt.figure(figsize=(8.27, 11.69), dpi=100)
                fig.subplots_adjust(left=0.07, right=0.93, top=0.92, bottom=0.07)
                plt.text(0.5, 0.965, title, ha="center", va="top", fontsize=16, fontweight="bold")
                y = 0.93
            plt.text(0.07, y, ln, ha="left", va="top", fontsize=10, family="monospace"); y -= line_h
        y -= 0.01
    if footer: plt.text(0.5, 0.045, footer, ha="center", va="center", fontsize=8, alpha=0.7)
    pdf.savefig(fig); plt.close(fig)

print("ARTIFACTS helper ready →", OUT)


ARTIFACTS helper ready → /kaggle/working


In [111]:
# === Build: ED_Research_System_Handoff_v1.pdf ===
handoff = OUT / "ED_Research_System_Handoff_v1.pdf"

sections = [
    ("ED Research System — Handoff (Freeze Snapshot)", [
        f"Timestamp: {NOW}",
        "Status: Phase 1 (Ops) — COMPLETE; Phase 2 (Clinical Integration) — COMPLETE.",
        "Purpose: Freeze current working state for safe transfer; ML remains advice-only."
    ]),
    ("Architecture & Invariants", [
        "Core: WorkflowState + TinyCritics; TrackerService (+ repos, QR, SOP); Streamlit UI (split); advice-only MLP bridge; Gate engine wired.",
        "Invariants: CONFIG frozen (only MODEL_BUNDLE_PATH may change); CSV/dir semantics; QR generate+manual-fallback; SOP auto-pull offline-safe; adjustable thresholds (equipment overdue, vitals overdue)."
    ]),
    ("Implemented (Evidence)", [
        "Tracker: moves logging; robust EquipmentRepository.read; movement analytics.",
        "QR: generate + manual fallback update path.",
        "SOPs: refresh_sop_registry() present; offline-safe summary.",
        "UI: split helpers; adjustable thresholds present.",
        "ML: bundle autodetect + load; advice_gatepos_mlp.csv written; calibrator-safe.",
        "Gate Engine: TTL_MAX_OK, GATES_WIRED_OK, PHASE2_BUNDLE_SMOKE_OK are OK."
    ]),
    ("How to Run (Safe Defaults)", [
        "Do not mutate frozen CONFIG; use overlay CFG for tests.",
        "Bundle: /kaggle/input/ed-pipeline-bundle-ui/ed_phase2_model_thr_patched(1).joblib",
        "MIMIC-IV-ED demo: /kaggle/input/mimic-iv-demo-v2-2/*.csv"
    ]),
    ("Paths (current run)", [
        "EQUIPMENT_STATUS_PATH: /kaggle/working/equipment_status.csv",
        "EQUIPMENT_MOVES_LOG_PATH: /kaggle/working/moves_log.csv",
        "SOP_REGISTRY_PATH: /kaggle/working/sop_registry.csv",
        "QR_OUTPUT_DIR: /kaggle/working/qrs",
        "EVENT_LOG_PATH: /kaggle/working/event_log.jsonl",
        "MODEL_BUNDLE_PATH: autodetected in /kaggle/working/*.joblib",
        "Advice outputs: /kaggle/working/advice_gatepos_mlp.csv"
    ]),
    ("Known Limits & Next Steps", [
        "Limits: demo subset; sklearn version warnings (acceptable); UI launch depends on Streamlit in runtime.",
        "Next: labels + multi-task MLP (advice-only) as refiners; per-head calibration; optional SOP telemetry."
    ]),
]

with PdfPages(handoff) as pdf:
    for title, paras in sections:
        add_pdf_page(pdf, title, paras, width=100, footer="Handoff • "+NOW)

print("Wrote:", handoff)


Wrote: /kaggle/working/ED_Research_System_Handoff_v1.pdf


In [112]:
# === Build: ED_Research_System_Summary_v1.pdf ===
summary = OUT / "ED_Research_System_Summary_v1.pdf"

paras = [
    f"Timestamp: {NOW}",
    "Phase 1 + Phase 2 COMPLETE (advice-only).",
    "Works: tracker/moves, QR+fallback, SOP offline-safe, UI thresholds, MLP advice bridge, gate engine sanity.",
    "Safe running: overlay CFG for tests; outputs under /kaggle/working.",
    "Next: clinical targets → multi-task heads + calibration."
]
with PdfPages(summary) as pdf:
    add_pdf_page(pdf, "ED Research System — Summary (Freeze Snapshot)", paras, width=102, footer="Summary • "+NOW)

print("Wrote:", summary)


Wrote: /kaggle/working/ED_Research_System_Summary_v1.pdf


In [113]:
# === Build: ChatGPT_Handoff_with_Contract_v1.pdf ===
chatgpt = OUT / "ChatGPT_Handoff_with_Contract_v1.pdf"

contract_lines = [
"Follow this contract exactly. When uncertain ask don't guess.",
"",
"Invariants (MUST NOT CHANGE/REMOVE)",
"• WorkflowState unchanged; exposes .feature_dict(), .update_state_from_event(...), .apply_event_log(...).",
"• TinyCritics uses WorkflowState.feature_dict(); cold-start safe.",
"• CONFIG bootstrap with defaults; flags RUN_UI=False, RUN_PIPELINE=False. Paths keep CSV/dir semantics.",
"• Tracker core first-class (tracker_core.TrackerService + tracker_ui.run_ui(...) or inlined eq.). No import-time side effects.",
"• SOP auto-pull present & offline-safe (refresh_sop_registry(...)); lazy imports; returns summary on failure.",
"• QR: generate + scan-to-update; manual payload fallback updates moves if decode libs absent.",
"• Overdue/alerts: equipment overdue (last_seen) + lingering patient (since_vitals_min) with adjustable thresholds in UI.",
"• Notebook metadata has kernelspec: python3.",
"",
"No-hallucinations / verification rules (MANDATORY)",
"• Only claim 'implemented/working' if code changed AND a local smoke/check passes.",
"• Cite provenance when copying cells from older notebooks.",
"• State uncertainty for external deps; don't imply success unless verified.",
"• No virtual resources: reference only existing files/URLs (or create explicitly).",
"• Provide exact diffs or full cells for modifications.",
"• No silent renames/moves; update all call sites.",
"• Ask before destructive edits.",
"",
"Allowed changes",
"• New features only behind guards: if RUN_PIPELINE: / if RUN_UI: .",
"• Append after core definitions; do not reorder WorkflowState/TinyCritics.",
"• Optional deps lazy-imported.",
"• You may extend feature_dict() by adding keys (no renames/removals).",
"",
"Forbidden changes",
"• No placeholders for WorkflowState; no renames of core classes/functions/paths.",
"• No top-level execution; no references before definition.",
"• Do not touch PDFs/data except via tracker/SOP services.",
"",
"Delivery checklist (MUST PASS, overlay suggested)",
"1 Overlay CFG with RUN_UI=False; RUN_PIPELINE=False.",
"2 TinyCritics cold-start; 2 actions; p in [0,1].",
"3 Tracker core: equipment_status(); log_move(); moves_log.csv exists; QR make(); SOP read() not None.",
"4 SOP auto-pull surface present; offline returns summary.",
"5 QR manual payload fallback updates move.",
"6 UI shows adjustable thresholds (minutes) for equipment overdue and vitals overdue.",
"7 Notebook metadata kernelspec: python3."
]

with PdfPages(chatgpt) as pdf:
    add_pdf_page(pdf, "ChatGPT Handoff — ED Research System (Freeze)", [
        f"Timestamp: {NOW}",
        "Scope: future assistants continue without breaking invariants; ML stays advice-only."
    ], footer="ChatGPT Handoff • "+NOW)
    add_pdf_page(pdf, "Contract", contract_lines, footer="Contract • "+NOW)
    add_pdf_page(pdf, "Operational Guidance", [
        "Keep changes behind guards; never mutate frozen CONFIG directly (use overlay CFG).",
        "Provide evidence (tiny smoke/file write) with any implementation claim.",
        "Maintain provenance; avoid virtual resources."
    ], footer="Guidance • "+NOW)
    add_pdf_page(pdf, "Paths & Semantics", [
        "EQUIPMENT_STATUS_PATH: /kaggle/working/equipment_status.csv (CSV)",
        "EQUIPMENT_MOVES_LOG_PATH: /kaggle/working/moves_log.csv (CSV)",
        "SOP_REGISTRY_PATH: /kaggle/working/sop_registry.csv (CSV)",
        "QR_OUTPUT_DIR: /kaggle/working/qrs (DIR)",
        "EVENT_LOG_PATH: /kaggle/working/event_log.jsonl (JSONL)",
        "MODEL_BUNDLE_PATH: autodetected (*.joblib)",
        "Advice outputs: /kaggle/working/advice_gatepos_mlp.csv"
    ], footer="Paths • "+NOW)

print("Wrote:", chatgpt)


Wrote: /kaggle/working/ChatGPT_Handoff_with_Contract_v1.pdf


In [114]:
# === Write tests/ + pyproject.toml + README_TESTS.md ===
from pathlib import Path
tests = OUT / "tests"
tests.mkdir(parents=True, exist_ok=True)

(tests / "conftest.py").write_text(
"""import pytest

def pytest_collection_modifyitems(config, items):
    pass
""", encoding="utf-8")

(tests / "test_equipment_repo.py").write_text(
"""import pandas as pd, pytest

try:
    from tracker_core import TrackerService
except Exception:
    pytest.skip('tracker_core not importable; run after packaging or with alias shim', allow_module_level=True)

def test_equipment_repo_read_and_move(tmp_path):
    CFG = {
        'EQUIPMENT_STATUS_PATH': str(tmp_path / 'equipment_status.csv'),
        'EQUIPMENT_MOVES_LOG_PATH': str(tmp_path / 'moves_log.csv'),
        'SOP_REGISTRY_PATH': str(tmp_path / 'sop_registry.csv'),
        'QR_OUTPUT_DIR': str(tmp_path / 'qrs'),
        'EVENT_LOG_PATH': str(tmp_path / 'event_log.jsonl'),
        'DATA_ROOT': str(tmp_path),
    }
    t = TrackerService.from_config(CFG)
    df = t.equipment_status()
    assert isinstance(df, pd.DataFrame)
    t.log_move('pump-TEST', 'A1', 'C3')
    m = pd.read_csv(CFG['EQUIPMENT_MOVES_LOG_PATH'])
    assert (m['equip_id'] == 'pump-TEST').any()
    assert (m['to'] == 'C3').any()
""", encoding="utf-8")

(tests / "test_qr_fallback.py").write_text(
"""import pandas as pd, pytest

try:
    from tracker_core import TrackerService
except Exception:
    pytest.skip('tracker_core not importable; run after packaging or with alias shim', allow_module_level=True)

def test_qr_manual_fallback_updates_move(tmp_path):
    CFG = {
        'EQUIPMENT_STATUS_PATH': str(tmp_path / 'equipment_status.csv'),
        'EQUIPMENT_MOVES_LOG_PATH': str(tmp_path / 'moves_log.csv'),
        'SOP_REGISTRY_PATH': str(tmp_path / 'sop_registry.csv'),
        'QR_OUTPUT_DIR': str(tmp_path / 'qrs'),
        'EVENT_LOG_PATH': str(tmp_path / 'event_log.jsonl'),
        'DATA_ROOT': str(tmp_path),
    }
    t = TrackerService.from_config(CFG)
    payload = 'id=pump-QR&x=1'
    equip_id = payload.split('id=',1)[1].split('&',1)[0]
    t.log_move(equip_id, '', 'ZZ')
    m = pd.read_csv(CFG['EQUIPMENT_MOVES_LOG_PATH'])
    assert (m['equip_id'] == equip_id).any()
    assert (m['to'] == 'ZZ').any()
""", encoding="utf-8")

(tests / "test_sop_autopull.py").write_text(
"""import pytest

try:
    from refresh_sop import refresh_sop_registry
except Exception:
    refresh_sop_registry = None

if refresh_sop_registry is None:
    pytest.skip('refresh_sop_registry not importable; run after extracting to a module', allow_module_level=True)

def test_sop_offline_safe(tmp_path):
    CFG = {
        'SOP_REGISTRY_PATH': str(tmp_path / 'sop_registry.csv'),
        'DATA_ROOT': str(tmp_path),
        'QR_OUTPUT_DIR': str(tmp_path / 'qrs'),
        'EQUIPMENT_STATUS_PATH': str(tmp_path / 'equipment_status.csv'),
        'EQUIPMENT_MOVES_LOG_PATH': str(tmp_path / 'moves_log.csv'),
        'EVENT_LOG_PATH': str(tmp_path / 'event_log.jsonl'),
    }
    res = refresh_sop_registry(CFG, base_url='http://127.0.0.1:9/offline')
    assert isinstance(res, dict)
""", encoding="utf-8")

(OUT / "pyproject.toml").write_text(
"""[build-system]
requires = ["setuptools>=61"]
build-backend = "setuptools.build_meta"

[tool.ruff]
line-length = 100
target-version = "py311"
select = ["E","F","I"]
ignore = ["E501"]
exclude = ["__results__.html", ".ipynb_checkpoints"]

[tool.mypy]
python_version = "3.11"
ignore_missing_imports = true
warn_unused_ignores = false
warn_redundant_casts = true
warn_unused_configs = true
exclude = ["^.*\\.ipynb$", "__results__\\.html", ".ipynb_checkpoints"]

[tool.pytest.ini_options]
addopts = "-q"
testpaths = ["tests"]
""", encoding="utf-8")

(OUT / "README_TESTS.md").write_text(
"""# Tests & Tooling (non-intrusive)
Place `tests/` and `pyproject.toml` at the repository root (same folder as your notebook).
They are safe: tests skip unless `tracker_core` / `refresh_sop_registry` are importable.

Run locally (optional):
    pip install pytest ruff mypy
    pytest -q
    ruff check .
    mypy .

Future packaging:
- ed_tracker.tracker_core: TrackerService, EquipmentRepository, MovesLogRepository, QRService, SOPRegistry
- ed_tracker.sop: refresh_sop_registry
""", encoding="utf-8")

print("Wrote tests/, pyproject.toml, README_TESTS.md under", OUT)


Wrote tests/, pyproject.toml, README_TESTS.md under /kaggle/working


In [115]:
# === List artifacts ===
from pathlib import Path
for p in sorted(Path("/kaggle/working").iterdir()):
    print("-", p)
print("Done. You can now Save Version (Commit).")


- /kaggle/working/.virtual_documents
- /kaggle/working/ChatGPT_Handoff_with_Contract_v1.pdf
- /kaggle/working/ED_Research_System_Handoff_v1.pdf
- /kaggle/working/ED_Research_System_Summary_v1.pdf
- /kaggle/working/README_TESTS.md
- /kaggle/working/advice_gatepos_mlp.csv
- /kaggle/working/ed_phase2_model_thr_patched(1).joblib
- /kaggle/working/edtracker_pkg
- /kaggle/working/equipment_status.csv
- /kaggle/working/interaction_rules.json
- /kaggle/working/moves_log.csv
- /kaggle/working/patient_allergies.json
- /kaggle/working/pyproject.toml
- /kaggle/working/qrs
- /kaggle/working/tests
Done. You can now Save Version (Commit).


In [116]:
def test_phase1_quickcheck_moves(tmp_path, monkeypatch):
    # arrange: point CONFIG to tmp paths, keep RUN_* false
    import ed_tracker.quickcheck as qc
    cfg = {**qc.CONFIG}
    for k in ("QR_OUTPUT_DIR","EVENT_LOG_PATH","SOP_REGISTRY_PATH",
              "EQUIPMENT_STATUS_PATH","EQUIPMENT_MOVES_LOG_PATH"):
        cfg[k] = str(tmp_path / k)
    # act
    res = qc.phase1_quickcheck(cfg)
    # assert
    assert {"moves","sop","ui_vitals_threshold"} <= res.keys()
    assert res["moves"] is True


In [117]:
def test_capacity_blocked_escalates():
    from ed_tracker.gatepacks import capacity_gatepack
    cfg = {"TH_ICU_CAP_OK": 0.25, "TH_ICU_CAP_TIGHT": 0.15}
    fd  = {"suspected_condition":"respiratory","cap_internal_medicine_icu":0.0,
           "ems_triage_code":1,"admit_decision":"ICU","Triage":"Orange"}
    g = set(capacity_gatepack(fd, cfg)["gates"])
    assert {"ICU_BLOCKED","ED_BOARDING_RISK","ALERT_NURSE","ALERT_DOC","TRANSFER_BLOCK_CAPACITY"} <= g


In [118]:
def test_ttl_merges_by_max():
    from ed_tracker.gatepacks import merge_gatepacks
    p1={"gates":["ALERT_NURSE"],"ttl":{"ALERT_NURSE":120}}
    p2={"gates":["ALERT_NURSE"],"ttl":{"ALERT_NURSE":300}}
    out = merge_gatepacks(p1,p2)
    assert out["ttl"]["ALERT_NURSE"] == 300


In [119]:
def test_pregnancy_years_ruleout_thresholds():
    from ed_tracker.calculators import assess_pregnancy_years
    ctx = {"pregnant": True, "pe_most_likely": False, "dvt_signs": False, "hemoptysis": False}
    labs= {"d_dimer": 900, "d_dimer_unit": "μg/L FEU"}   # below 1000
    assert assess_pregnancy_years(ctx, labs)["rule_out"] is True
